# מערכת תמיכה חכמה - מכללת עזריאלי
## IT Support Agent with NiceGUI + LangGraph + Supabase
פרויקט תכנות בסביבת בינה מלאכותית גנרטיבית

In [ ]:
# תא 1 - התקנת חבילות
!pip -q install nicegui langchain langchain-openai langgraph nest_asyncio
!pip -q install tavily-python
!pip -q install langchain-chroma chromadb
!pip -q install pypdf pymupdf
!pip -q install langchain-community langchain-text-splitters
!pip -q install langchain-mcp-adapters
!pip -q install supabase
print("✅ All packages installed")

In [ ]:
# תא 2 - uvicorn 0.29.0 (חייב להיות אחרי תא 1)
!pip -q install uvicorn==0.29.0
print("✅ uvicorn 0.29.0 installed")

In [ ]:
# תא 3 - הורדת Cloudflare tunnel
!rm -f /usr/local/bin/cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("✅ cloudflared installed")

In [ ]:
# תא 4 - חיבור Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
# תא 5 - טעינת מפתחות
import os

# OpenAI
with open('/content/drive/MyDrive/openai_api_key.txt', 'r') as f:
    os.environ['OPENAI_API_KEY'] = f.read().strip()

# Tavily
with open('/content/drive/MyDrive/tavily.txt', 'r') as f:
    os.environ['TAVILY_API_KEY'] = f.read().strip()

# Supabase MCP Token (for agent)
with open('/content/drive/MyDrive/supabaseMCP.txt', 'r') as f:
    os.environ['SUPABASE_TOKEN'] = f.read().strip()

# Supabase Project URL
with open('/content/drive/MyDrive/supabase_url.txt', 'r') as f:
    url = f.read().strip()
    if not url.startswith('http'):
        url = 'https://' + url
    if not url.endswith('.supabase.co'):
        url = url + '.supabase.co'
    os.environ['SUPABASE_URL'] = url

# Supabase Service Role Key (for UI direct access)
with open('/content/drive/MyDrive/supabase_service_key.txt', 'r') as f:
    os.environ['SUPABASE_SERVICE_KEY'] = f.read().strip()

# LangSmith
with open('/content/drive/MyDrive/langsmith-api-key.txt', 'r') as f:
    os.environ['LANGCHAIN_API_KEY'] = f.read().strip()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'azrieli-it-agent'

print("✅ OpenAI API Key loaded")
print("✅ Tavily API Key loaded")
print("✅ Supabase MCP Token loaded")
print(f"✅ Supabase URL: {os.environ['SUPABASE_URL']}")
print(f"✅ Supabase Service Key: {os.environ['SUPABASE_SERVICE_KEY'][:20]}...")
print("✅ LangSmith connected - project: azrieli-it-agent")

In [ ]:
# תא 6 - nest_asyncio
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio applied")

In [ ]:
# תא 7 - Cloudflare tunnel
import subprocess, time, re

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8090'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

for _ in range(20):
    line = tunnel.stderr.readline()
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group()
            break
    time.sleep(1)

In [ ]:
# תא 8 - Imports
import os, asyncio, glob, base64
from typing import List, TypedDict, Literal, Annotated
from uuid import uuid4

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import fitz  # pymupdf
from tavily import TavilyClient
from pydantic import BaseModel, Field
from supabase import create_client
from nicegui import ui, app

print("✅ All imports loaded")

In [ ]:
# תא 9 - טעינת PDFs ובניית מאגר ידע
kb_path = '/content/drive/MyDrive/KB_Guides/'
images_path = '/content/kb_images/'
os.makedirs(images_path, exist_ok=True)

all_docs = []
pdf_files = glob.glob(os.path.join(kb_path, '*.pdf'))

for pdf_file in pdf_files:
    filename = os.path.basename(pdf_file)
    print(f"📄 Loading: {filename}")
    # Extract clean guide name for embedding enrichment
    guide_name = filename.replace('.pdf', '').replace('מערכת קריאות- מחלקת מחשוב - ', '').replace(' - מאגר הידע', '').strip()
    loader = PyPDFLoader(pdf_file)
    pages = loader.load()
    for page in pages:
        page.metadata['source_file'] = filename
        page.metadata['guide_name'] = guide_name
        page.metadata['image_path'] = os.path.join(images_path, f"{filename}_page_{page.metadata.get('page', 0)}.png")
        # Prepend guide name to content - this dramatically improves search matching
        page.page_content = f"מדריך: {guide_name}\n\n{page.page_content}"
    all_docs.extend(pages)
    pdf_doc = fitz.open(pdf_file)
    for i, fitz_page in enumerate(pdf_doc):
        pix = fitz_page.get_pixmap(dpi=150)
        pix.save(os.path.join(images_path, f"{filename}_page_{i}.png"))
    pdf_doc.close()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunks = splitter.split_documents(all_docs)
embeddings = OpenAIEmbeddings()
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"\n✅ Loaded {len(pdf_files)} PDFs, {len(chunks)} chunks (enriched with guide names)")

In [ ]:
# תא 10 - הגדרת כלים

# Mapping from PDF keywords to actual URLs on the support site
KB_URL_MAP = {
    "איפוס סיסמא": "https://support.jce.ac.il/helpdesk/KB/View/73",
    "איפוס סיסמה": "https://support.jce.ac.il/helpdesk/KB/View/73",
    "שינוי סיסמא": "https://support.jce.ac.il/helpdesk/KB/View/72",
    "שינוי סיסמה": "https://support.jce.ac.il/helpdesk/KB/View/72",
    "כללי סיסמא": "https://support.jce.ac.il/helpdesk/KB/View/87",
    "פתיחת חשבון": "https://support.jce.ac.il/helpdesk/KB/View/71",
    "אימות דו": "https://support.jce.ac.il/helpdesk/KB/View/3635",
    "MFA": "https://support.jce.ac.il/helpdesk/KB/View/3635",
    "שיתוף קבצים": "https://support.jce.ac.il/helpdesk/KB/View/80",
    "OneDrive": "https://support.jce.ac.il/helpdesk/KB/View/80",
    "העברת הודעות": "https://support.jce.ac.il/helpdesk/KB/View/79",
    "FindTime": "https://support.jce.ac.il/helpdesk/KB/View/107",
    "לוח שנה": "https://support.jce.ac.il/helpdesk/KB/View/279",
    "מערכת הקריאות": "https://support.jce.ac.il/helpdesk/KB/View/85",
    "שליטה למחשב": "https://support.jce.ac.il/helpdesk/KB/View/3923",
    "תמיכה מרחוק": "https://support.jce.ac.il/helpdesk/KB/View/3923",
    "הדפסה": "https://support.jce.ac.il/helpdesk/KB",
    "מדפסת": "https://support.jce.ac.il/helpdesk/KB",
    "eduroam": "https://support.jce.ac.il/helpdesk/KB",
    "WiFi": "https://support.jce.ac.il/helpdesk/KB",
    "VPN": "https://support.jce.ac.il/helpdesk/KB",
    "Moodle": "https://support.jce.ac.il/helpdesk/KB",
}

def get_guide_url(source_filename):
    """Find the actual URL for a guide based on its filename"""
    for keyword, url in KB_URL_MAP.items():
        if keyword in source_filename:
            return url
    return None  # No specific URL found - let retrieve tool provide the URL

@tool
def retrieve(query: str) -> str:
    """Search the Azrieli College IT knowledge base for guides and solutions."""
    results = vector_store.similarity_search_with_score(query, k=4)
    if not results:
        return "No relevant guides found in knowledge base."
    output = []
    seen_sources = set()
    for doc, score in results:
        source = doc.metadata.get('source_file', 'unknown')
        if source in seen_sources:
            continue
        seen_sources.add(source)
        page = doc.metadata.get('page', 0)
        image_path = doc.metadata.get('image_path', '')
        guide_name = source.replace('.pdf', '').replace('מערכת קריאות- מחלקת מחשוב - ', '').replace(' - מאגר הידע', '').strip()
        guide_url = get_guide_url(source)
        if guide_url is None:
            guide_url = "https://support.jce.ac.il/helpdesk/KB"
        relevance = "HIGH" if score < 0.8 else "MEDIUM" if score < 1.3 else "LOW"
        output.append(
            f"Guide: {guide_name} | URL: {guide_url} | Source: {source} | Page: {page} | Image: {image_path} | Relevance: {relevance}\n"
            f"Content: {doc.page_content}"
        )
    return "\n\n---\n\n".join(output[:3]) if output else "No relevant guides found in knowledge base."

@tool
def tavily_search(query: str) -> str:
    """Search the internet for current IT support information. Use Hebrew or English queries."""
    client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
    response = client.search(query, max_results=3, search_depth="basic")
    results = []
    for r in response.get('results', []):
        results.append(f"Title: {r['title']}\nURL: {r.get('url','')}\nContent: {r['content']}")
    return "\n\n---\n\n".join(results) if results else "No results found on the internet."

# --- ADMIN TOOLS ---
@tool
def list_tickets(status_filter: str = "all") -> str:
    """List all tickets from Supabase. Admin only. status_filter can be: all, open, progress, done, waiting"""
    try:
        query = supabase_client.table("tickets").select("*").order("created_at", desc=True)
        if status_filter and status_filter != "all":
            query = query.eq("status", status_filter)
        tickets = query.limit(20).execute().data or []
        if not tickets:
            return "אין קריאות שירות במערכת."
        lines = []
        for t in tickets:
            lines.append(f"#{t.get('ticket_number','')} | {t.get('subject','')} | סטטוס: {t.get('status','')} | עדיפות: {t.get('priority','')} | סטודנט: {t.get('student_name','')} | קטגוריה: {t.get('category','')}")
        return "\n".join(lines)
    except Exception as e:
        return f"שגיאה בטעינת קריאות: {str(e)[:80]}"

@tool
def update_ticket(ticket_number: str, new_status: str) -> str:
    """Update the status of a ticket. Admin only. new_status must be: open, progress, done, waiting"""
    valid = ["open", "progress", "done", "waiting"]
    if new_status not in valid:
        return f"סטטוס לא תקין. הערכים האפשריים: {', '.join(valid)}"
    try:
        result = supabase_client.table("tickets").update({"status": new_status, "last_update": f"סטטוס עודכן ל-{new_status}"}).eq("ticket_number", ticket_number).execute()
        if result.data:
            # Log the activity
            status_heb = {"open":"פתוח","progress":"בטיפול","done":"נפתר","waiting":"ממתין"}.get(new_status, new_status)
            supabase_client.table("activity_log").insert({"text": f"קריאה {ticket_number} עודכנה ל-{status_heb}", "icon_type": "green"}).execute()
            return f"קריאה {ticket_number} עודכנה בהצלחה לסטטוס: {status_heb}"
        return f"לא נמצאה קריאה עם מספר {ticket_number}"
    except Exception as e:
        return f"שגיאה בעדכון: {str(e)[:80]}"

@tool
def delete_ticket(ticket_number: str) -> str:
    """Delete a ticket from the system. Admin only."""
    try:
        result = supabase_client.table("tickets").delete().eq("ticket_number", ticket_number).execute()
        if result.data:
            supabase_client.table("activity_log").insert({"text": f"קריאה {ticket_number} נמחקה", "icon_type": "red"}).execute()
            return f"קריאה {ticket_number} נמחקה בהצלחה."
        return f"לא נמצאה קריאה עם מספר {ticket_number}"
    except Exception as e:
        return f"שגיאה במחיקה: {str(e)[:80]}"

print("✅ Tools: retrieve, tavily_search, list_tickets, update_ticket, delete_ticket")

In [ ]:
# תא 11 - LLM, State, System Message
llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    is_admin: bool
    awaiting_hitl: bool
    ticket_draft: dict
    request_type: str       # Router output: password/network/office/moodle/printing/admin/general
    complexity: str         # Router output: simple/medium/complex
    confidence_score: int   # Critic output: 1-5 quality score for the response
    retry_count: int        # Critic: how many times answer was sent back for improvement

SYSTEM_MESSAGE = """You are the IT Support Agent for Azrieli College of Engineering (עזריאלי - מכללה אקדמית להנדסה) in Jerusalem.

PERSONALITY: Friendly, professional, empathetic. Like a helpful friend who knows IT. Never cold or robotic.
LANGUAGE: ALWAYS respond in Hebrew. Use clear, simple language that any student can understand.

TOOLS (use in this order):
1. retrieve - Search college IT knowledge base. ALWAYS try this FIRST. Results include a Relevance field (HIGH/MEDIUM/LOW). IGNORE results marked as LOW relevance - they are not related to the question.
2. tavily_search - Search the internet. Use ONLY if retrieve returned no relevant results or "No relevant guides found". When you get results, ALWAYS mention the source name and include the URL in your response like this: "לפי [שם האתר](URL)"
3. list_tickets / update_ticket / delete_ticket - Admin only tools.
4. ProposeTicket - Open a support ticket when you can't solve the problem.

TERMINAL COMMANDS:
For ANY computer/technical issue (slow computer, freezing, crashes, disk full, network problems, printing issues, etc.):
1. FIRST: Use tavily_search to find general solutions and tips. Present them with sources.
2. SECOND: Try to understand the specific problem with follow-up questions.
3. LAST: At the END of your message, say: "בנוסף, אני יכול להציע לך פקודות שתוכל להריץ בטרמינל כדי לאבחן ולטפל בבעיה. רוצה שאציע?"
4. When user says YES:
   - First explain: "מעולה! אנחנו הולכים לעבוד עם שורת הפקודה (CMD). כדי לפתוח אותה כמנהל מערכת: לחץ Win+X ובחר Terminal (Admin) או חפש CMD בתפריט התחל, לחץ עליו ימני ובחר הפעל כמנהל."
   - Then give ONLY ONE command in triple backticks
   - Explain what the command does
   - Ask: "הריצת את הפקודה? מה הופיע?"
5. Based on the result, give the NEXT relevant command.
- NEVER give multiple commands at once
- ALWAYS explain what each command does BEFORE showing it
- Format commands in triple backticks (```)

WHEN TO USE TAVILY (web search):
- When retrieve returns no relevant guides
- For general tech questions (slow computer, recommendations, troubleshooting)
- For questions about software/hardware not specific to the college
- ALWAYS cite the source: "לפי [שם האתר](URL)"
- Show at least 2-3 practical tips from the search results

SCOPE:
- Answer ANY question related to computers, technology, software, hardware, networking, and IT
- This includes personal computer issues, buying recommendations, general tech advice
- Only decline questions completely unrelated to technology (e.g., cooking, sports)

IMPORTANT: For EVERY new question, you MUST call the retrieve tool again. Do NOT reuse results from previous questions - each question needs a fresh search.

RESPONSE STRATEGY:
- If the question is SPECIFIC (e.g., "איך מאפסים סיסמא", "איך מגדירים אימות דו-שלבי") → search retrieve immediately and give the answer.
- If the question is VAGUE or GENERAL (e.g., "WiFi לא עובד", "המחשב לא עובד", "יש בעיה במייל") → DO NOT search yet. First ask 1-2 SHORT clarifying questions. Examples:
  * "WiFi לא עובד" → "באיזה קמפוס אתה? מה בדיוק קורה - אין רשת בכלל או שיש חיבור בלי אינטרנט?"
  * "המחשב איטי" → Ask "מחשב של המכללה או אישי?" then use tavily_search for tips, give advice, and at the end offer terminal commands
  * "בעיה במייל" → "לא מצליח להיכנס? לא מקבל מיילים? יש הודעת שגיאה?"
- After clarification, search with a PRECISE query based on the answer.

RESPONSE FORMAT - Follow this structure:
1. Start with a brief empathetic acknowledgment (1 sentence, e.g., "אני מבין שזה מתסכל, בוא נפתור את זה")
2. Give the solution in clear numbered steps
3. If you used a guide, cite it: "לפי המדריך: [שם המדריך](URL)"
4. End with: "אם זה לא עזר, אני יכול לפתוח קריאת שירות"

CRITICAL RULES:
- NEVER invent URLs - only use URLs from the retrieve tool results
- If retrieve returns guides NOT related to the question, IGNORE them completely. Don't mention irrelevant guides.
- If no relevant guide exists: help from general IT knowledge first, then use tavily_search
- Be concise - max 200 words per response. Students want quick solutions, not lectures.
- Use numbered steps (1, 2, 3) not bullet points
- When using tavily_search results, always mention the source: "לפי [שם האתר](URL)"

WHEN TO OPEN A TICKET:
- User explicitly asks to open a ticket
- User says the solution didn't help (after 2+ attempts)
- Your confidence is 3 or below (internal, never show to user)
- Use ProposeTicket with Hebrew subject, category (סיסמאות/Office/רשת/Moodle/הדפסה/כללי), priority (קריטית/דחופה/בינונית/נמוכה - MUST be in Hebrew), description
- Do NOT use ProposeTicket for admin operations!

ADMIN MODE:
- User writes "אני מנהל" -> ask for 4-digit code
- Correct code: 1234 -> confirm admin access, use admin tools
- Admin can: list_tickets, update_ticket, delete_ticket

SCREENSHOTS:
- The UI automatically shows guide screenshots when relevant. Do NOT try to show/link/reference images yourself.

RULES:
- Stay focused on computers, technology, and IT - decline only non-tech questions
- NEVER fabricate information
- Be patient and respectful
- If you truly don't know, say so honestly
"""

class ProposeTicket(BaseModel):
    """Propose opening a new IT support ticket. Only use for NEW tickets, NOT for admin operations."""
    subject: str = Field(description="Ticket subject in Hebrew")
    category: str = Field(description="Category")
    priority: str = Field(description="Priority in Hebrew ONLY. Must be exactly one of: קריטית / דחופה / בינונית / נמוכה")
    description: str = Field(description="Description in Hebrew")

print("✅ LLM (gpt-4o), State, System Message, ProposeTicket ready")

In [ ]:
# תא 12 - בניית הגרף עם HITL + Admin Tools
def build_graph(mcp_tools=None):
    local_tools = [retrieve, tavily_search, list_tickets, update_ticket, delete_ticket]
    all_tools = local_tools + (mcp_tools or [])
    llm_with_tools = llm.bind_tools(all_tools + [ProposeTicket])
    tool_node = ToolNode(all_tools)

    # --- Router/Planner Node: classifies request type and complexity ---
    def router_node(state: AgentState) -> AgentState:
        """Analyze the user message and classify request type, complexity, and admin status."""
        # Extract the last user message text
        user_text = ""
        for msg in reversed(state["messages"]):
            if isinstance(msg, HumanMessage):
                content = msg.content if isinstance(msg.content, str) else str(msg.content)
                user_text = content.lower()
                break

        # Classify request type by keywords
        request_type = "general"
        kw_map = {
            "password": ["סיסמ", "סיסמא", "סיסמה", "password", "נעול", "חשבון", "התחברות", "login"],
            "network": ["wifi", "eduroam", "vpn", "אינטרנט", "רשת", "חיבור"],
            "office": ["outlook", "מייל", "office", "onedrive", "teams", "אימות דו", "mfa", "דוא"],
            "moodle": ["moodle", "מודל", "קורס", "הגשה"],
            "printing": ["הדפס", "מדפסת", "printer"],
            "admin": ["מנהל", "קריאות", "טיקט", "tickets"],
        }
        for rtype, keywords in kw_map.items():
            if any(kw in user_text for kw in keywords):
                request_type = rtype
                break

        # Determine complexity based on message length and question marks
        msg_len = len(user_text)
        has_question = "?" in user_text or "?" in user_text
        if msg_len < 30 and not has_question:
            complexity = "simple"
        elif msg_len > 120 or user_text.count("?") > 1 or user_text.count("?") > 1:
            complexity = "complex"
        else:
            complexity = "medium"

        # Check admin status from state
        is_admin = state.get("is_admin", False)
        if "מנהל" in user_text or "admin" in user_text:
            is_admin = True

        print(f"🔀 Router: type={request_type}, complexity={complexity}, admin={is_admin}")
        return {"request_type": request_type, "complexity": complexity, "is_admin": is_admin}

    def answer_node(state: AgentState) -> AgentState:
        messages = state["messages"]
        sys = SystemMessage(content=SYSTEM_MESSAGE)
        # Clean history: keep recent tool messages (current cycle) but remove old ones
        # This forces the agent to search fresh for each new question
        cleaned = []
        # Find the last HumanMessage index - everything after it is "current cycle"
        last_human_idx = -1
        for i, m in enumerate(messages):
            if isinstance(m, HumanMessage):
                last_human_idx = i
        for i, m in enumerate(messages):
            if isinstance(m, SystemMessage):
                continue
            if i >= last_human_idx:
                # Current cycle - keep everything (including tool calls/results)
                cleaned.append(m)
            else:
                # Old messages - keep only Human and AI text responses
                if isinstance(m, HumanMessage):
                    cleaned.append(m)
                elif isinstance(m, AIMessage):
                    if m.content and not (hasattr(m, 'tool_calls') and m.tool_calls):
                        # Keep AI text responses, skip AI messages that are just tool calls
                        cleaned.append(m)
                # Skip old ToolMessages entirely
        all_msgs = [sys] + cleaned
        response = llm_with_tools.invoke(all_msgs)
        return {"messages": [response]}

    def human_review(state: AgentState) -> AgentState:
        return state

    def finalize(state: AgentState) -> AgentState:
        last_ai = None
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and hasattr(msg, "tool_calls") and msg.tool_calls:
                last_ai = msg
                break
        if last_ai:
            for tc in last_ai.tool_calls:
                if tc["name"] == "ProposeTicket":
                    return {"messages": [ToolMessage(tool_call_id=tc["id"], content="Ticket approved.", name=tc["name"])], "awaiting_hitl": False}
        return {"awaiting_hitl": False}

    def route_after_answer(state: AgentState) -> Literal["tools", "human_review", "critic"]:
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                if tc["name"] == "ProposeTicket":
                    return "human_review"
            return "tools"
        return "critic"

    # --- Critic/Reflector Node: validates response quality before sending to user ---
    def critic_node(state: AgentState) -> AgentState:
        """Check the quality of the agent's response. Send back to answer if insufficient."""
        last_msg = state["messages"][-1] if state["messages"] else None
        retry_count = state.get("retry_count", 0)
        complexity = state.get("complexity", "simple")

        # Extract response text
        response_text = ""
        if isinstance(last_msg, AIMessage) and last_msg.content:
            response_text = last_msg.content

        # Quality checks
        has_content = len(response_text.strip()) > 10
        is_long_enough = len(response_text) > 40 or complexity == "simple"

        # Score the response (1-5)
        score = 5
        if not has_content:
            score = 1
        elif not is_long_enough and complexity == "complex":
            score = 2
        elif complexity == "complex" and len(response_text) < 100:
            score = 3

        print(f"🔍 Critic: score={score}, retry={retry_count}, complexity={complexity}, len={len(response_text)}")

        # If score is too low and we haven't retried too many times, send back
        if score <= 2 and retry_count < 2:
            print(f"🔄 Critic: sending back to answer (score={score}, retry={retry_count})")
            return {"confidence_score": score, "retry_count": retry_count + 1}

        return {"confidence_score": score}

    def route_after_critic(state: AgentState) -> Literal["answer", "__end__"]:
        """If critic gave low score and retries remain, loop back to answer."""
        score = state.get("confidence_score", 5)
        retry_count = state.get("retry_count", 0)
        if score <= 2 and retry_count <= 2:
            return "answer"
        return "__end__"

    graph_builder = StateGraph(AgentState)
    graph_builder.add_node("router", router_node)
    graph_builder.add_node("answer", answer_node)
    graph_builder.add_node("tools", tool_node)
    graph_builder.add_node("human_review", human_review)
    graph_builder.add_node("finalize", finalize)
    graph_builder.add_node("critic", critic_node)
    graph_builder.add_edge(START, "router")
    graph_builder.add_edge("router", "answer")
    graph_builder.add_conditional_edges("answer", route_after_answer)
    graph_builder.add_edge("tools", "answer")
    graph_builder.add_edge("human_review", "finalize")
    graph_builder.add_edge("finalize", END)
    graph_builder.add_conditional_edges("critic", route_after_critic)
    checkpointer = MemorySaver()
    return graph_builder.compile(checkpointer=checkpointer, interrupt_before=["human_review"])

print("✅ build_graph() ready (with admin tools)")

In [ ]:
# תא 13 - הדפסת הגרף
from IPython.display import Image, display
try:
    temp_graph = build_graph(mcp_tools=[])
    display(Image(temp_graph.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
    print("✅ Graph displayed")
except:
    print("⚠️ Could not display graph (mermaid.ink timeout) - continuing...")

In [ ]:
# תא 14 - חיבור ישיר ל-Supabase + נתונים סטטיים
supabase_client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

def fetch_tickets(filter_status=None, student_name=None):
    try:
        query = supabase_client.table("tickets").select("*").order("created_at", desc=True)
        if filter_status and filter_status != 'all':
            query = query.eq("status", filter_status)
        if student_name:
            query = query.eq("student_name", student_name)
        return query.execute().data or []
    except Exception as e:
        print(f"Error fetching tickets: {e}")
        return []

def fetch_chat_sessions():
    try:
        return supabase_client.table("chat_sessions").select("*").order("created_at", desc=True).execute().data or []
    except:
        return []

def fetch_activity_log(limit=5):
    try:
        return supabase_client.table("activity_log").select("*").order("created_at", desc=True).limit(limit).execute().data or []
    except:
        return []

KB_ITEMS = [
    {"id": 1, "title": "מדריך לאיפוס סיסמא", "cat": "סיסמאות", "views": 1247, "tags": ["סיסמא", "איפוס"], "url": "https://support.jce.ac.il/helpdesk/KB/View/73"},
    {"id": 2, "title": "אימות דו-שלבי ב-Office 365", "cat": "Office", "views": 892, "tags": ["MFA", "אימות"], "url": "https://support.jce.ac.il/helpdesk/KB/View/3635"},
    {"id": 3, "title": "חיבור לרשת eduroam", "cat": "רשת", "views": 2103, "tags": ["WiFi", "eduroam"], "url": "https://support.jce.ac.il/helpdesk/KB"},
    {"id": 4, "title": "העברת הודעות אוטומטית ב-Outlook", "cat": "Office", "views": 534, "tags": ["outlook"], "url": "https://support.jce.ac.il/helpdesk/KB/View/79"},
    {"id": 5, "title": "הדפסה ממחשב אישי במעבדות", "cat": "הדפסה", "views": 1856, "tags": ["הדפסה"], "url": "https://support.jce.ac.il/helpdesk/KB"},
    {"id": 6, "title": "מדריך שימוש ב-Moodle", "cat": "Moodle", "views": 3201, "tags": ["Moodle", "קורסים"], "url": "https://support.jce.ac.il/helpdesk/KB"},
    {"id": 7, "title": "שיתוף קבצים ב-OneDrive", "cat": "Office", "views": 678, "tags": ["OneDrive"], "url": "https://support.jce.ac.il/helpdesk/KB/View/80"},
    {"id": 8, "title": "התקנת VPN לגישה מרחוק", "cat": "רשת", "views": 945, "tags": ["VPN"], "url": "https://support.jce.ac.il/helpdesk/KB"},
]

TOPIC_GREETINGS = {
    "password": "שלום! 🔑\nהבנתי שיש לך בעיה הקשורה לסיסמא או להתחברות.\nמה בדיוק קורה? למשל:\n- שכחת סיסמא?\n- החשבון נעול?\n- הסיסמא לא מתקבלת?",
    "office": "שלום! 📧\nאני כאן לעזור עם Office 365 והמייל של המכללה.\nמה הבעיה? למשל:\n- בעיה בכניסה למייל?\n- אימות דו-שלבי?\n- בעיה ב-OneDrive?",
    "wifi": "שלום! 📶\nאני כאן לעזור עם חיבור לאינטרנט ורשת במכללה.\nמה קורה? למשל:\n- לא מצליח להתחבר ל-eduroam?\n- האינטרנט איטי?\n- VPN לא עובד?",
    "moodle": "שלום! 📚\nאני כאן לעזור עם מערכת Moodle.\nמה הבעיה? למשל:\n- לא רואה קורס?\n- לא מצליח להגיש עבודה?\n- בעיה בכניסה?",
    "print": "שלום! 🖨️\nאני כאן לעזור עם הדפסה במכללה.\nמה קורה? למשל:\n- לא מדפיס?\n- בעיה בחשבון הדפסה?\n- מדפסת לא מזוהה?",
    "other": "שלום! 💻\nאני כאן לעזור עם כל בעיה טכנית.\nתאר/י לי את הבעיה ואני אנסה לעזור.",
}

try:
    result = supabase_client.table("tickets").select("*").limit(1).execute()
    print(f"✅ Supabase connected - {len(result.data)} rows in tickets")
except Exception as e:
    print(f"⚠️ Supabase: {e}")

print("✅ KB items, topic greetings ready")

In [ ]:
# תא 15 - בדיקת הסוכן
test_graph = build_graph(mcp_tools=[])
cfg = {"configurable": {"thread_id": "test_full"}}
result = test_graph.invoke(
    {"messages": [HumanMessage(content="איך מאפסים סיסמא במכללת עזריאלי?")],
     "is_admin": False, "awaiting_hitl": False, "ticket_draft": {},
     "request_type": "", "complexity": "", "confidence_score": 0, "retry_count": 0}, cfg)
print("Q: איך מאפסים סיסמא?")
print(f"A: {result['messages'][-1].content[:300]}")
print(f"Tools used: {sum(1 for m in result['messages'] if hasattr(m, 'tool_calls') and m.tool_calls)}")

print(f"\n{'='*50}")
print(f"🌐 הלינק הציבורי שלך:")
print(f"👉 {tunnel_url}")
print(f"{'='*50}\n")

In [ ]:
# תא 16 - האפליקציה המלאה (NiceGUI) - גרסה משודרגת
graph_instance = None
try:
    app.add_static_files('/kb_images', '/content/kb_images/')
except:
    pass  # Already added

@ui.page('/')
async def main_page():
    global graph_instance
    if graph_instance is None:
        graph_instance = build_graph(mcp_tools=[])

    # Persistent login via app.storage.user
    stored = app.storage.user
    logged_in = stored.get('logged_in', False)
    student_name = stored.get('student_name', '')
    student_phone = stored.get('student_phone', '')

    state = {
        'role': 'student', 'page': 'home', 'chat_topic': None,
        'chat_messages': [], 'thread_id': str(uuid4()), 'is_admin': False,
        'pending_ticket': None, 'pending_tc_id': None, 'session_created': False,
        'student_name': student_name, 'student_phone': student_phone,
        'response_times': [], 'ticket_filter': 'all',
        'logged_in': logged_in, 'busy': False, 'needs_render': False,
        'saved_chats': {'student': None, 'admin': None},
    }

    ui.add_head_html('<link href="https://fonts.googleapis.com/css2?family=Heebo:wght@300;400;500;600;700;800;900&display=swap" rel="stylesheet">')

    ui.add_css("""
        *{font-family:'Heebo',sans-serif!important;box-sizing:border-box}
        body{background:#F7F8FA!important;margin:0}
        .nicegui-content{padding:0!important;max-width:100%!important}
        .q-page-container{padding-top:0!important}
        .q-page{padding:0!important}
        .q-layout{min-height:0!important}
        #c0,#c1,#c2,#c3{padding:0!important;margin:0!important}
        .row,.column{padding:0!important;margin:0!important}
        ::-webkit-scrollbar{width:5px}
        ::-webkit-scrollbar-thumb{background:#D1D5DB;border-radius:3px}
        .sidebar{width:240px;height:100vh;position:fixed;right:0;top:0;z-index:100;background:linear-gradient(180deg,#1A1A2E,#16213E);box-shadow:-4px 0 24px rgba(0,0,0,.15);display:flex;flex-direction:column;direction:rtl}
        .sidebar-header{padding:24px 20px 18px;border-bottom:1px solid rgba(255,255,255,.08);display:flex;align-items:center;gap:12px}
        .sidebar-logo{min-width:54px;height:36px;border-radius:8px;background:#fff;display:flex;align-items:center;justify-content:center;box-shadow:0 2px 8px rgba(196,30,58,.3);padding:4px 6px;font-weight:800;color:#C41E3A;font-size:13px}
        .sidebar-menu{padding:14px 10px;flex:1;overflow-y:auto}
        .sidebar-menu-label{color:rgba(255,255,255,.3);font-size:10px;font-weight:600;padding:8px 14px;letter-spacing:1px}
        .menu-btn{width:100%;display:flex;align-items:center;gap:11px;padding:11px 14px;border-radius:10px;border:none;cursor:pointer;margin-bottom:2px;font-size:13.5px;font-family:inherit;background:transparent;color:rgba(255,255,255,.55);font-weight:400;direction:rtl;text-align:right;transition:background .15s}
        .menu-btn:hover{background:rgba(255,255,255,.05)}
        .menu-btn.active{background:rgba(196,30,58,.15);color:#FF6B81;font-weight:600}
        .menu-badge{margin-right:auto;background:#C41E3A;color:#fff;font-size:10px;font-weight:700;padding:2px 7px;border-radius:10px}
        .sidebar-bottom{padding:14px;border-top:1px solid rgba(255,255,255,.08)}
        .role-switcher{display:flex;gap:4px;margin-bottom:10px;background:rgba(255,255,255,.06);border-radius:8px;padding:3px}
        .role-btn{flex:1;padding:7px 0;border-radius:6px;border:none;cursor:pointer;font-size:11.5px;font-weight:600;font-family:inherit;background:transparent;color:rgba(255,255,255,.4)}
        .role-btn.active-student{background:rgba(255,255,255,.15);color:#fff}
        .role-btn.active-admin{background:#C41E3A;color:#fff}
        .user-info{display:flex;align-items:center;gap:10px;padding:10px 12px;background:rgba(255,255,255,.04);border-radius:10px}
        .user-avatar{width:34px;height:34px;border-radius:10px;display:flex;align-items:center;justify-content:center;color:#fff;font-weight:700;font-size:13px}
        .main-content{position:absolute;top:0;left:0;right:240px;padding:24px 28px;direction:rtl;min-height:100vh}
        .chat-container{display:flex;gap:20px;height:calc(100vh - 80px)}
        .chat-main{flex:1;display:flex;flex-direction:column;background:#fff;border-radius:16px;box-shadow:0 1px 3px rgba(0,0,0,.06);border:1px solid #E5E7EB;overflow:hidden}
        .chat-header{padding:16px 24px;border-bottom:1px solid #E5E7EB;display:flex;align-items:center;gap:12px}
        .chat-avatar{width:40px;height:40px;border-radius:12px;background:linear-gradient(135deg,#C41E3A,#E84563);display:flex;align-items:center;justify-content:center;color:#fff;font-size:20px}
        .chat-messages{flex:1;overflow-y:auto;padding:24px;background:#FAFBFC;display:flex;flex-direction:column;gap:16px}
        .msg-row{display:flex;gap:10px;align-items:flex-start}
        .msg-row.user{flex-direction:row-reverse}
        .msg-avatar{width:34px;height:34px;border-radius:10px;flex-shrink:0;display:flex;align-items:center;justify-content:center;font-size:16px}
        .msg-avatar.bot{background:linear-gradient(135deg,#C41E3A,#E84563);color:#fff}
        .msg-avatar.user{background:#E5E7EB;color:#6B7280}
        .msg-bubble{padding:14px 18px;max-width:75%;font-size:13.5px;line-height:1.7;white-space:pre-wrap;box-shadow:0 1px 4px rgba(0,0,0,.06)}
        .msg-bubble.bot{background:#fff;color:#1A1A2E;border-radius:4px 14px 14px 14px}
        .msg-bubble.user{background:linear-gradient(135deg,#C41E3A,#D4253F);color:#fff;border-radius:14px 4px 14px 14px}
        .chat-input-area{padding:16px 20px;border-top:1px solid #E5E7EB;display:flex;gap:10px;align-items:center}
        .chat-input{flex:1;padding:11px 16px;border-radius:12px;border:1px solid #E5E7EB;font-size:13.5px;outline:none;font-family:inherit;direction:rtl}
        .chat-input:focus{border-color:#C41E3A}
        .send-btn{width:42px;height:42px;border-radius:12px;border:none;background:#C41E3A;color:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;box-shadow:0 2px 8px rgba(196,30,58,.4);font-size:18px}
        .icon-btn{width:40px;height:40px;border-radius:10px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;color:#6B7280;font-size:18px}
        .faq-box{background:#fff;border-radius:14px;padding:18px;border:1px solid #E5E7EB}
        .faq-btn{width:100%;text-align:right;padding:9px 12px;border-radius:8px;border:1px solid #F3F4F6;background:#FAFAFA;cursor:pointer;font-size:12.5px;margin-bottom:6px;color:#1A1A2E;font-family:inherit;direction:rtl}
        .faq-btn:hover{border-color:#C41E3A;background:#FFF5F5}
        .typing-dot{width:7px;height:7px;border-radius:50%;background:#D1D5DB;display:inline-block;animation:pulse 1.2s infinite}
        .typing-dot:nth-child(2){animation-delay:.2s}
        .typing-dot:nth-child(3){animation-delay:.4s}
        @keyframes pulse{0%,100%{opacity:1}50%{opacity:.3}}
        .hero{background:linear-gradient(135deg,#1A1A2E,#16213E,#0F3460);border-radius:20px;padding:48px 44px;margin-bottom:28px;position:relative;overflow:hidden;color:#fff;width:100%}
        .hero h1{font-size:30px;font-weight:800;margin-bottom:12px;line-height:1.3}
        .hero p{color:rgba(255,255,255,.6);font-size:15px;max-width:520px;line-height:1.7;margin-bottom:28px}
        .btn-red{padding:12px 28px;border-radius:12px;border:none;background:#C41E3A;color:#fff;font-size:14px;font-weight:600;cursor:pointer;box-shadow:0 4px 16px rgba(196,30,58,.4);font-family:inherit}
        .btn-outline{padding:12px 28px;border-radius:12px;border:1px solid rgba(255,255,255,.2);background:rgba(255,255,255,.06);color:#fff;font-size:14px;font-weight:500;cursor:pointer;font-family:inherit}
        .card{background:#fff;border-radius:14px;padding:24px;box-shadow:0 1px 3px rgba(0,0,0,.06);border:1px solid #E5E7EB}
        .badge{display:inline-flex;align-items:center;gap:6px;padding:4px 12px;border-radius:8px;font-size:12px;font-weight:600}
        .hitl-panel{padding:18px;background:linear-gradient(135deg,#FEF2F2,#FFF1F2);border-top:3px solid #DC2626;direction:rtl}
        .hitl-header{display:flex;align-items:center;justify-content:space-between;margin-bottom:14px}
        .hitl-details{background:#fff;border-radius:12px;padding:16px;margin-bottom:14px;border:1px solid #FECACA;display:grid;grid-template-columns:100px 1fr;gap:10px 14px;font-size:13px}
        .hitl-label{font-weight:600;color:#6B7280}
        .hitl-approve{padding:11px 24px;border-radius:10px;border:none;background:#059669;color:#fff;font-size:13px;font-weight:600;cursor:pointer;box-shadow:0 2px 8px rgba(5,150,105,.3);font-family:inherit}
        .hitl-reject{padding:11px 24px;border-radius:10px;border:none;background:#FEE2E2;color:#DC2626;font-size:13px;font-weight:500;cursor:pointer;font-family:inherit}
        .suggest-btn{padding:7px 14px;border-radius:8px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;font-size:12px;font-family:inherit;color:#374151;transition:all .15s}
        .suggest-btn:hover{border-color:#C41E3A;background:#FFF5F5;color:#C41E3A}
        .rate-btn{width:30px;height:30px;border-radius:8px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;display:inline-flex;align-items:center;justify-content:center;font-size:14px;transition:all .15s}
        .rate-btn:hover{background:#F3F4F6;transform:scale(1.15)}
        .kb-card{background:#fff;border-radius:16px;padding:24px;border:1px solid #E5E7EB;transition:all .2s;position:relative;overflow:hidden}
        .kb-card:hover{box-shadow:0 8px 25px rgba(0,0,0,.08);transform:translateY(-2px)}
        .toast{position:fixed;bottom:24px;left:50%;transform:translateX(-50%);padding:12px 24px;border-radius:12px;font-size:13px;font-weight:500;z-index:9999;box-shadow:0 4px 20px rgba(0,0,0,.15);font-family:'Heebo',sans-serif;animation:toastIn .3s ease}
        @keyframes toastIn{from{opacity:0;transform:translateX(-50%) translateY(20px)}to{opacity:1;transform:translateX(-50%) translateY(0)}}
    """)

    app_container = ui.html('').style('position:fixed;top:0;left:0;right:0;bottom:0;width:100vw;height:100vh;overflow-y:auto;padding:0;margin:0;z-index:50;background:#F7F8FA;')

    # ===== HTML BUILDERS =====
    def sidebar_html():
        role, page = state['role'], state['page']
        if role == 'admin':
            items = [('home','ראשי','🏠'),('chat','סוכן AI','💬'),('tickets','קריאות שירות','🎫'),('kb','מאגר ידע','📚')]
        else:
            items = [('home','ראשי','🏠'),('chat',"צ'אט עם הסוכן",'💬'),('kb','מדריכים','📚'),('my-tickets','הקריאות שלי','🎫')]
        m = ''
        try:
            open_tickets = len(fetch_tickets(filter_status='open'))
            my_tickets = len(fetch_tickets(student_name=state['student_name']))
        except:
            open_tickets = 0; my_tickets = 0
        for pid, label, icon in items:
            act = 'active' if page == pid else ''
            bdg = ''
            if pid == 'tickets' and role == 'admin' and open_tickets > 0:
                bdg = f'<span class="menu-badge">{open_tickets}</span>'
            if pid == 'my-tickets' and my_tickets > 0:
                bdg = f'<span class="menu-badge" style="background:#3B82F6">{my_tickets}</span>'
            m += f'<button class="menu-btn {act}" data-action="nav" data-value="{pid}"><em style="font-style:normal;font-size:18px;width:24px;text-align:center">{icon}</em>{label}{bdg}</button>'
        sc = 'active-student' if role == 'student' else ''
        ac = 'active-admin' if role == 'admin' else ''
        abg = 'linear-gradient(135deg,#C41E3A,#E84563)' if role == 'admin' else 'linear-gradient(135deg,#3B82F6,#60A5FA)'
        at = 'מנ' if role == 'admin' else state['student_name'][:2]
        un = 'מנהל מערכת' if role == 'admin' else state['student_name']
        us = 'Admin' if role == 'admin' else "סטודנט"
        return f"""<div class="sidebar">
            <div class="sidebar-header"><div class="sidebar-logo"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAACwAAAAeCAIAAADYVGvwAAABCGlDQ1BJQ0MgUHJvZmlsZQAAeJxjYGA8wQAELAYMDLl5JUVB7k4KEZFRCuwPGBiBEAwSk4sLGHADoKpv1yBqL+viUYcLcKakFicD6Q9ArFIEtBxopAiQLZIOYWuA2EkQtg2IXV5SUAJkB4DYRSFBzkB2CpCtkY7ETkJiJxcUgdT3ANk2uTmlyQh3M/Ck5oUGA2kOIJZhKGYIYnBncAL5H6IkfxEDg8VXBgbmCQixpJkMDNtbGRgkbiHEVBYwMPC3MDBsO48QQ4RJQWJRIliIBYiZ0tIYGD4tZ2DgjWRgEL7AwMAVDQsIHG5TALvNnSEfCNMZchhSgSKeDHkMyQx6QJYRgwGDIYMZAKbWPz9HbOBQAAAGkklEQVR4nO2WbVBU1xnHn+ecc/fuLsvuQngTEhFBrTFJm87EGDGjzUyq0xmcfihSm5p8qUJ8ac3IMMZIxE4CSaEt6hgbx6rRvHSqMa2mZTCaTx21adpqbMGKishbgF1g2d277L33nKcfrhBpiEkNM/mS59s5c855fud5+Z+DRARftbGvGgDga4hP7GuIMbsjiKluqP8PgpQCUoAIRCTlVNHcFoIIiEApUMqZQMYA2WhXFyAi5w7N2Mo7h0D1vxdC5JOQKdsGpeKtl7pfOzTY3ORf8Mi0J54ILirmLhcpQoZwM06EjH0CBwCIk1/vVpeTKCYRKYWcS8OQhpHo6BD+gJaT07J6deS9U0x3cY9HJkeVpVy5uYUvvpD1vWWJzk49N4+N0xNN8D0+JCKHcqKJzn37ZTjE3G5QZIbD/K70gmc29v2pqauxkXFhDQ1aQ4MoRH5lZdHz1TcCwdCJ4zJhkCJP/vTcn6z2Fs68tGFj+PTplDmzvbOKtMzM7BWl3vz8RMcNEQxwjxddGiISAEnFOENEACDblskkWbY1EuG+VOHOzWl5vhqSSaa5AEgmk/bAQOG26qGTzT0HDrqzs7imkW1feW5L+tKl+es2pC1Z0nfk98FFi7KWLzeuXmtdU55obRV+38i5s6F3380rL2ea1rp2/dCpU66cbO7zCX9Axo2Ubz9YtH1bov36lZoaFY2RaUrTtEJhnp5276u/QSIKNTW3rFnNOUMuAMEMh7PLVs7+VUN7Q0PXr3cwTQDjIujngWDm8u/PfG6zNC3u0q6+UNu+/efC59EzMu1YDD3eWfX1wYULWysqImf+oqWmKksiZ8lQyPetB+9//TAiXvzxqvi/L3KPBxkn2yZdf+B3bwUXLEBl28h57+E32hvquaaRZaMm7Fgs+Nhj976yu//o0eFz5wIPzfcWFTGXK9FxPfze6ejf/3HXsqXZZSvM3r7eNw4Pvv9+yuw5s+p/MdrddfmZTXZoQAsEnKqywuHA4iX3HX7NjkQ+WlE2erlNBAOgCJS0LXvuvr0Zjz9OUiI5fYhoRUas4SEVN5yhOTSk4jERTAvOf6j32B86G35p9n+s4nFUxDy6NEZFVnbm8pJ71q9DTdjhwZ6DB7sP7NfcHqbrjopYI9GslT+cu3NH/MrVfz35pNXRwX0+p4Msw5jTuCOnrJSkRM6F00UkpRbwJ/v727ZstXu7CJgdi0ojLuNGRknJ9A0bpj9d0bFr52hkmHm8dtxguu7Om+bKybaHhyNnznbu3p280aGnpwMAINhGHLlWsL1m+rq1kb9+0FJeYQ/0C4cAwIrFi16qyykrJWkjFxNalJRCxiIf/O0/P9toXGrRMzKcxrIiEVfe3TM2b077zpKBPx7vO3IkuODhjJISLSszcuZM99598Y8uCJ+P6ToAkJRWJOKdN29WXV1w4SN9x95pq6qCZJJ7PDcJRqIza2ruWVvhxGASnXBkxxoevrz52dDbb2upqSgEAKhk0orG/MXFBZWbvIVFRvu1UFNTqOnPZscN7vVyr9fZbo9EwO2ZtmrVzC3PMo/nWt1L3bt2cV1HIYgIEc1IZMbW6vyfrr+VYBKxcuIBAJ2v7u2or6d4XPhTSREi2iMjxJg+LS/Z3QmWJXw+1HUiAqWkESdFgUcfza+qDM5/ONHd01ZVNXTypBYIjJ9sRkYKqich+GzFJELGoucvtG2tjp47q/n9KIQzryyLuVyI6JQYABBCygPfzFtTnrnsuwDw8ZGj7S/WWr09WiBAUjpaaY1EC7bXTH/ayQIDwM+DcEikRM6laXbu3tO1Zw9FhkXADwQ3u8nZjCgtK+tHK2dsquQp3ug/z3cfOBA6cYLrOtfdJG1gDJSyYrHCbTV3T6yDLwRxa2piLa3tL7882NzMheBe73gMAICA0KUzj5ekbQ8OgmkKv58IgCQwDkrZo4nC2tq8p576LILPgbiZGqWczX3Hj3c17oxdvCi8HjZWDQAASpFSAIiaQMZISgBAIVTStM3knB2NOaWlStrMkYM7gXBMKUJAZNJI9Bw61PPb/aPX27nbzXT9008iAJCUdjTK/P6i2rqcFT8Y14MvBzF2tBMScyDU++Zbfe8cS169RqMJAEAhAAEISEoiYqmpwcWLZ1RuSp037zZZuBMIgAnZsQ0jev5C9MMPR9uvJ/v6lGWiEK6sbO/cb6QVF6fefx/cUlVTCvEplNssgcn+L1MHMe5n7PFDREAEAiIFRIgIX8z9l4aYOvsvHNe4qHyvwYIAAAAASUVORK5CYII=" style="height:28px;width:auto"/></div><div><div style="color:#fff;font-size:12px;font-weight:600">עזריאלי מכללה אקדמית להנדסה</div></div></div>
            <div class="sidebar-menu"><div class="sidebar-menu-label">תפריט</div>{m}</div>
            <div class="sidebar-bottom">
                <div class="role-switcher"><button class="role-btn {sc}" data-action="role" data-value="student">סטודנט</button><button class="role-btn {ac}" data-action="role" data-value="admin">מנהל</button></div>
                <div class="user-info"><div class="user-avatar" style="background:{abg}">{at}</div><div><div style="color:#fff;font-size:12.5px;font-weight:500">{un}</div><div style="color:rgba(255,255,255,.35);font-size:10.5px">{us}</div></div></div>
            </div>
        </div>"""

    def clean_agent_response(text):
        """Clean markdown and convert to HTML"""
        import re as _r
        from urllib.parse import urlparse
        if not text: return text

        # Step 1: Convert markdown links [text](url) to placeholder (before HTML escape)
        links_found = []
        def _save_link(m):
            label = m.group(1).strip('_ ')
            url = m.group(2).strip('_ ')
            idx = len(links_found)
            links_found.append((label, url))
            return f'%%LINK{idx}%%'
        text = _r.sub(r'\[_*([^\]]+?)_*\]\(_*(https?://[^)\s]+?)_*\)', _save_link, text)

        # Step 2: Clean __ around standalone URLs
        text = _r.sub(r'__\s*(https?://[^\s]+?)\s*__', r'\1', text)
        text = text.replace('__', '')

        # Step 3: Save bare URLs as placeholders too
        def _save_bare(m):
            url = m.group(0)
            idx = len(links_found)
            try:
                domain = urlparse(url).netloc
            except:
                domain = url[:30]
            links_found.append((domain or url[:30], url))
            return f'%%LINK{idx}%%'
        text = _r.sub(r'https?://[^\s\)]+', _save_bare, text)

        # Step 3.5: Extract code blocks (```...```) BEFORE HTML escape
        code_blocks = []
        def _save_code_block(m):
            code = m.group(1).strip()
            idx = len(code_blocks)
            code_blocks.append(code)
            return f'%%CODE{idx}%%'
        text = _r.sub(r'```(?:\w*\n)?(.+?)```', _save_code_block, text, flags=_r.DOTALL)

        # Step 4: HTML escape (safe now - no URLs or tags left)
        text = text.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

        # Step 5: Bold **text**
        text = _r.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)

        # Step 5.3: Headers ###
        text = _r.sub(r'(?m)^#{2,3}\s*(.+?)$', r'<div style="font-size:14px;font-weight:700;color:#1A1A2E;margin:14px 0 6px">\1</div>', text)
        text = _r.sub(r'#{2,3}\s+(\S[^\n]{2,30})', r'<div style="font-size:14px;font-weight:700;color:#1A1A2E;margin:14px 0 6px">\1</div>', text)

        # Step 5.5: Inline code `command` - clean card style
        text = _r.sub(r'`([^`]+)`', r'<code style="background:#F1F5F9;color:#0F172A;padding:3px 10px;border-radius:6px;font-family:Consolas,monospace;font-size:12.5px;direction:ltr;display:inline-block;border:1px solid #E2E8F0;font-weight:600">\1</code>', text)

        # Step 6: Numbered lists
        text = _r.sub(r'(?m)^(\d+)\.\s', r'<strong>\1.</strong> ', text)

        # Step 7: Bullet points
        text = _r.sub(r'(?m)^-\s', '• ', text)

        # Step 8: Newlines
        text = text.replace('\n', '<br>')

        # Step 8.5: Restore code blocks as clean command cards
        for i, code in enumerate(code_blocks):
            esc_code = code.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
            # Clean command for data attribute (escape quotes)
            clean_cmd = code.strip().replace('"', '&quot;').replace("'", '&#39;')
            lines = [l for l in esc_code.split('\n') if l.strip()]
            cmd_text = '<br>'.join(lines)
            block_html = f'''<div data-cmd="{clean_cmd}" style="margin:10px 0;border-radius:12px;overflow:hidden;border:1px solid #E2E8F0;box-shadow:0 1px 4px rgba(0,0,0,.06)">
                <div style="padding:16px 20px;background:#F8FAFC;display:flex;align-items:center;gap:14px">
                    <div style="width:32px;height:32px;border-radius:8px;background:#EFF6FF;display:flex;align-items:center;justify-content:center;flex-shrink:0;font-size:16px">⌨️</div>
                    <div style="flex:1;font-family:Consolas,\'Courier New\',monospace;font-size:14px;color:#0F172A;direction:ltr;text-align:left;font-weight:600;letter-spacing:.3px;padding:8px 14px;background:#fff;border-radius:8px;border:1px solid #E2E8F0">{cmd_text}</div>
                    <div style="color:#fff;font-size:12px;cursor:pointer;padding:8px 18px;background:linear-gradient(135deg,#3B82F6,#2563EB);border-radius:10px;font-weight:600;white-space:nowrap;font-family:inherit;box-shadow:0 2px 6px rgba(37,99,235,.3);display:flex;align-items:center;gap:4px" data-action="copy-code" data-value="{i}">📋 העתק</div>
                </div>
            </div>'''
            text = text.replace(f'%%CODE{i}%%', block_html)

        # Step 9: Restore links as styled cards with favicon
        # Deduplicate: track URLs already shown
        seen_urls = set()
        for i, (label, url) in enumerate(links_found):
            # Skip duplicate URLs
            if url in seen_urls:
                text = text.replace(f'%%LINK{i}%%', '')
                continue
            seen_urls.add(url)
            try:
                domain = urlparse(url).netloc
            except:
                domain = ''
            favicon = f'https://www.google.com/s2/favicons?domain={domain}&sz=16' if domain else ''
            fav_html = f'<img src="{favicon}" style="width:16px;height:16px;border-radius:2px" onerror="this.style.display=\'none\'"/> ' if favicon else ''
            # Label: specific guide vs general KB
            is_general_kb = 'support.jce.ac.il' in url and '/KB/View/' not in url
            if is_general_kb:
                label = 'מאגר ידע כללי'
            link_html = f'<a href="{url}" target="_blank" style="display:inline-flex;align-items:center;gap:6px;padding:4px 10px;background:#EFF6FF;border:1px solid #BFDBFE;border-radius:8px;text-decoration:none;color:#1D4ED8;font-size:12.5px;font-weight:500;margin:2px 0">{fav_html}{label} ↗</a>'
            if is_general_kb:
                link_html = f'<div style="font-size:12.5px;color:#6B7280;margin:4px 0">לנוחיותך, מצורף קישור למאגר הידע הכללי:</div>' + link_html
            text = text.replace(f'%%LINK{i}%%', link_html)

        return text

    def msgs_html():
        import re as _re
        from urllib.parse import quote
        h = ''
        for idx, msg in enumerate(state['chat_messages']):
            r = msg['role']
            av = '<div class="msg-avatar bot">🤖</div>' if r == 'bot' else '<div class="msg-avatar user">👤</div>'
            c = msg.get('content', '')
            if c == '...':
                bubble = '<span class="typing-dot"></span><span class="typing-dot"></span><span class="typing-dot"></span>'
            elif r == 'bot':
                bubble = clean_agent_response(c)
            else:
                c = c.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;').replace('\n','<br>')
                bubble = c
            if msg.get('is_hitl'):
                ticket = msg.get('ticket_data', {})
                pri_map = {'קריטית':'קריטית 🔴','דחופה':'דחופה 🟠','בינונית':'בינונית 🟡','נמוכה':'נמוכה 🟢','high':'קריטית 🔴','med':'בינונית 🟡','low':'נמוכה 🟢'}
                cat_map = {'סיסמאות':'סיסמאות וכניסה','Office':'דוא"ל ו-Office','רשת':'אינטרנט ורשת','Moodle':'Moodle','הדפסה':'הדפסה','כללי':'כללי','חיבור מרחוק':'חיבור מרחוק'}
                pri_heb = pri_map.get(ticket.get('priority','med'), ticket.get('priority','בינונית'))
                cat_heb = cat_map.get(ticket.get('category',''), ticket.get('category',''))
                h += f"""<div style="width:100%"><div style="padding:20px;background:linear-gradient(135deg,#FEF2F2,#FFF8F0);border-radius:16px;border:1px solid #FED7AA;direction:rtl">
                    <div style="display:flex;align-items:center;gap:10px;margin-bottom:16px">
                        <div style="width:36px;height:36px;border-radius:10px;background:#FEF3C7;display:flex;align-items:center;justify-content:center;font-size:20px">🤖</div>
                        <div><div style="font-size:15px;font-weight:700;color:#1A1A2E">לא הצלחתי לפתור את הבעיה</div>
                        <div style="font-size:12px;color:#6B7280">ממליץ לפתוח קריאת שירות. בדוק את הפרטים ואשר:</div></div>
                    </div>
                    <div style="background:#fff;border-radius:12px;padding:18px;margin-bottom:16px;border:1px solid #F3F4F6">
                        <div style="display:grid;grid-template-columns:80px 1fr;gap:12px 16px;font-size:13.5px">
                            <span style="color:#6B7280;font-weight:600">📋 נושא</span><span style="font-weight:500;color:#1A1A2E">{ticket.get('subject','')}</span>
                            <span style="color:#6B7280;font-weight:600">📁 תחום</span><span>{cat_heb}</span>
                            <span style="color:#6B7280;font-weight:600">⚡ דחיפות</span><span style="font-weight:600">{pri_heb}</span>
                            <span style="color:#6B7280;font-weight:600">📝 תיאור</span><span style="color:#374151">{ticket.get('description','')}</span>
                        </div>
                    </div>
                    <div style="display:flex;gap:10px;flex-wrap:wrap">
                        <button style="padding:12px 28px;border-radius:10px;border:none;background:#059669;color:#fff;font-size:14px;font-weight:600;cursor:pointer;box-shadow:0 2px 8px rgba(5,150,105,.3);font-family:inherit" data-action="send" data-value="מאשר">✅ כן, פתח קריאה</button>
                        <button style="padding:12px 28px;border-radius:10px;border:none;background:#EFF6FF;color:#1E40AF;font-size:13px;font-weight:500;cursor:pointer;font-family:inherit" data-action="edit-ticket" data-value="1">✏️ ערוך קריאה</button>
                    </div>
                </div></div>"""
                continue
            h += f'<div class="msg-row {r}">{av}<div class="msg-bubble {r}">{bubble}'
            # Carousel for guide images
            if msg.get('guide_images') and len(msg['guide_images']) > 0:
                images = msg['guide_images']
                img_source = msg.get('image_source', 'מדריך')
                uid = str(abs(hash(str(images))))[-6:]
                img_urls = ['/kb_images/' + quote(os.path.basename(img)) for img in images]
                total = len(img_urls)
                imgs_js = str(img_urls).replace("'", '"')
                h += f"""<div style="margin-top:10px;padding:12px;background:#F9FAFB;border-radius:10px;border:1px solid #E5E7EB">
                    <div style="font-size:12px;color:#6B7280;margin-bottom:8px;display:flex;justify-content:space-between"><span>📸 {img_source}</span><span id="counter_{uid}">עמוד 1 מתוך {total}</span></div>
                    <div style="text-align:center"><img id="carousel_{uid}" src="{img_urls[0]}" style="max-width:100%;border-radius:8px;cursor:zoom-in;max-height:400px" data-action="zoom" data-value="{uid}" onerror="this.style.display='none'"/></div>"""
                if total > 1:
                    h += f'<div style="display:flex;justify-content:center;gap:12px;margin-top:10px"><button style="padding:6px 16px;border-radius:8px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;font-family:inherit;font-size:13px" data-action="carousel-prev" data-value="{uid}">→ הקודם</button><button style="padding:6px 16px;border-radius:8px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;font-family:inherit;font-size:13px" data-action="carousel-next" data-value="{uid}">הבא ←</button></div>'
                h += f'<div data-carousel-id="{uid}" data-carousel-imgs="{",".join(img_urls)}" style="display:none"></div></div>'
            if msg.get('user_image'):
                h += f'<div style="margin-top:10px"><img src="{msg["user_image"]}" style="max-width:100%;border-radius:8px;max-height:250px;cursor:zoom-in" data-action="zoom-single" data-value="{msg["user_image"]}"/></div>'
            # Suggestion buttons - INSIDE the bubble (only for last bot message)
            if r == 'bot' and c != '...' and not msg.get('is_hitl') and idx == len(state['chat_messages']) - 1:
                suggestions = msg.get('suggestions', ["תסביר יותר 📝", "תפתח קריאה 🎫"])
                sug_btns = ''.join(f'<button class="suggest-btn" data-action="send" data-value="{s}">{s}</button>' for s in suggestions)
                h += f'<div style="margin-top:12px;padding-top:10px;border-top:1px solid #F3F4F6">'
                h += f'<div style="display:flex;gap:6px;flex-wrap:wrap">{sug_btns}</div>'
                h += '</div>'
            h += '</div></div>'
        return h

    def chat_html():
        m = msgs_html()
        faqs = ["איך מאפסים סיסמא?","WiFi לא עובד","בעיה ב-Moodle","איך מדפיסים?","אימות דו-שלבי"]
        fh = ''.join(f'<button class="faq-btn" data-action="send" data-value="{q}">{q}</button>' for q in faqs)
        return f"""<div class="main-content"><div class="chat-container">
            <div class="chat-main">
                <div class="chat-header">
                    <div class="chat-avatar">🤖</div>
                    <div style="flex:1"><div style="font-size:14px;font-weight:600">סוכן IT חכם</div><div style="font-size:12px;color:#10B981">🟢 פעיל</div></div>
                    <button style="padding:6px 14px;border-radius:8px;border:1px solid #E5E7EB;background:#fff;cursor:pointer;font-size:12px;font-family:inherit;color:#6B7280" data-action="newchat" data-value="1">🔄 שאלה חדשה</button>
                </div>
                <div class="chat-messages" id="chatMessages">{m}</div>
                <div class="chat-input-area">
                    <button class="icon-btn" data-action="camera" data-value="1" title="צלם תמונה">📷</button>
                    <button class="icon-btn" data-action="upload" data-value="1" title="העלה תמונה">🖼️</button>
                    <input type="file" id="camInput" accept="image/*" capture="camera" style="display:none"/>
                    <input type="file" id="imgInput" accept="image/*" style="display:none"/>
                    <input class="chat-input" id="chatInput" placeholder="תאר את הבעיה או שאל שאלה..."/>
                    <button class="send-btn" data-action="sendchat" data-value="1" title="שלח">➤</button>
                </div>
            </div>
            <div style="width:250px"><div class="faq-box"><div style="font-size:13px;font-weight:600;margin-bottom:14px;color:#6B7280">שאלות נפוצות</div>{fh}</div></div>
        </div></div>"""

    def home_admin_html():
        tix = fetch_tickets()
        th = ''
        if not tix:
            th = '<div style="padding:20px;text-align:center;color:#9CA3AF;font-size:13px">📭 אין קריאות עדיין</div>'
        else:
            bm = {"open":("פתוח","#FEF3C7","#92400E","#F59E0B"),"progress":("בטיפול","#DBEAFE","#1E40AF","#3B82F6"),"done":("נפתר","#D1FAE5","#065F46","#10B981"),"waiting":("ממתין","#FEE2E2","#991B1B","#EF4444")}
            for i, t in enumerate(tix[:4]):
                status = t.get('status','open')
                b = bm.get(status, bm['open']); bd = 'border-bottom:1px solid #F3F4F6;' if i < min(len(tix),4)-1 else ''
                th += f'<div style="display:flex;justify-content:space-between;align-items:center;padding:12px 0;{bd}"><div><div style="font-size:13px;font-weight:500">{t.get("subject","")}</div><div style="font-size:11px;color:#9CA3AF">{t.get("student_name","")} · {t.get("ticket_number","")}</div></div><span class="badge" style="background:{b[1]};color:{b[2]}"><span style="width:6px;height:6px;border-radius:50%;background:{b[3]};display:inline-block"></span>{b[0]}</span></div>'
        kh = ''
        cat_icons = {"סיסמאות":"🔑","Office":"📧","רשת":"📶","Moodle":"📚","הדפסה":"🖨️"}
        cat_colors = {"סיסמאות":("#FEF3C7","#92400E"),"Office":("#DBEAFE","#1E40AF"),"רשת":("#D1FAE5","#065F46"),"Moodle":("#EDE9FE","#5B21B6"),"הדפסה":("#FCE7F3","#9D174D")}
        for i, k in enumerate(KB_ITEMS[:4]):
            cc = cat_colors.get(k["cat"], ("#F3F4F6","#374151"))
            icon = cat_icons.get(k["cat"], "📄")
            bd = 'border-bottom:1px solid #F3F4F6;' if i < 3 else ''
            kh += f'<a href="{k.get("url","#")}" target="_blank" style="text-decoration:none;color:inherit;display:flex;align-items:center;gap:12px;padding:12px 0;{bd}"><div style="width:38px;height:38px;border-radius:10px;background:{cc[0]};display:flex;align-items:center;justify-content:center;font-size:18px;flex-shrink:0">{icon}</div><div style="flex:1"><div style="font-size:13px;font-weight:600;color:#1A1A2E">{k["title"]}</div><div style="font-size:11px;color:#9CA3AF">{k["cat"]} · {k["views"]:,} צפיות</div></div><span style="font-size:12px;color:#C41E3A">↗</span></a>'
        return f"""<div class="main-content">
            <div class="hero"><div style="display:flex;align-items:center;gap:10px;margin-bottom:14px"><div style="width:36px;height:36px;border-radius:10px;background:rgba(196,30,58,.2);display:flex;align-items:center;justify-content:center">⚡</div><span style="color:#FF6B81;font-size:13px;font-weight:600">AI-Powered IT Support</span></div><h1>מערכת תמיכה חכמה<br/>עזריאלי - מכללה אקדמית להנדסה</h1><p>סוכן AI חכם שפותר בעיות טכניות בזמן אמת, מציג מדריכים ויזואליים, ופותח קריאות שירות רק כשבאמת צריך.</p><div style="display:flex;gap:12px"><button class="btn-red" data-action="nav" data-value="chat">💬 התחל שיחה</button><button class="btn-outline" data-action="nav" data-value="kb">עיין במדריכים</button></div></div>
            <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px"><div class="card"><div style="display:flex;justify-content:space-between;margin-bottom:18px"><h3 style="font-size:16px;font-weight:700">קריאות אחרונות</h3><button style="font-size:12px;color:#C41E3A;font-weight:600;background:none;border:none;cursor:pointer" data-action="nav" data-value="tickets">הצג הכל ←</button></div>{th}</div><div class="card"><div style="display:flex;justify-content:space-between;margin-bottom:18px"><h3 style="font-size:16px;font-weight:700">מדריכים פופולריים</h3><button style="font-size:12px;color:#C41E3A;font-weight:600;background:none;border:none;cursor:pointer" data-action="nav" data-value="kb">הצג הכל ←</button></div>{kh}</div></div>
        </div>"""

    def home_student_html():
        topics = [("🔑","סיסמא / התחברות","איפוס, שינוי, נעילת חשבון","#FEF3C7","password"),("📧","Office 365 / מייל","Outlook, OneDrive, Teams","#DBEAFE","office"),("📶","WiFi / רשת","eduroam, VPN, אינטרנט","#D1FAE5","wifi"),("📚","Moodle","גישה לקורסים, הגשת עבודות","#EDE9FE","moodle"),("🖨️","הדפסה","הדפסה במעבדות","#FCE7F3","print"),("💻","אחר","בעיה שלא מופיעה כאן","#F3F4F6","other")]
        th = ''.join(f'<button style="background:#fff;border-radius:14px;padding:20px;border:1px solid #E5E7EB;cursor:pointer;text-align:right;display:flex;align-items:flex-start;gap:14px;font-family:inherit" data-action="topic" data-value="{tp}"><div style="width:44px;height:44px;border-radius:12px;background:{co};display:flex;align-items:center;justify-content:center;font-size:22px;flex-shrink:0">{ic}</div><div><div style="font-size:14px;font-weight:600;color:#1A1A2E">{ti}</div><div style="font-size:12px;color:#6B7280">{de}</div></div></button>' for ic,ti,de,co,tp in topics)
        cat_icons2 = {"סיסמאות":"🔑","Office":"📧","רשת":"📶","Moodle":"📚","הדפסה":"🖨️"}
        cat_colors2 = {"סיסמאות":("#FEF3C7","#92400E","#F59E0B"),"Office":("#DBEAFE","#1E40AF","#3B82F6"),"רשת":("#D1FAE5","#065F46","#10B981"),"Moodle":("#EDE9FE","#5B21B6","#8B5CF6"),"הדפסה":("#FCE7F3","#9D174D","#EC4899")}
        kh = ''
        for k in KB_ITEMS[:4]:
            cc = cat_colors2.get(k["cat"], ("#F3F4F6","#374151","#6B7280"))
            icon = cat_icons2.get(k["cat"], "📄")
            kh += f'<a href="{k.get("url","#")}" target="_blank" style="text-decoration:none;color:inherit;display:block"><div class="kb-card" style="display:flex;align-items:center;gap:14px"><div style="width:46px;height:46px;border-radius:12px;background:linear-gradient(135deg,{cc[0]},{cc[0]}dd);display:flex;align-items:center;justify-content:center;font-size:22px;flex-shrink:0;box-shadow:0 2px 6px {cc[2]}25">{icon}</div><div style="flex:1"><div style="font-size:13.5px;font-weight:600;color:#1A1A2E">{k["title"]}</div><div style="display:flex;align-items:center;gap:8px;margin-top:4px"><span style="padding:2px 8px;border-radius:5px;background:{cc[0]};font-size:10.5px;color:{cc[1]};font-weight:500">{k["cat"]}</span><span style="font-size:10.5px;color:#9CA3AF">👁 {k["views"]:,}</span></div></div><span style="font-size:12px;color:#C41E3A;font-weight:600">צפה ↗</span></div></a>'
        sn = state['student_name'].split(' ')[0]
        return f"""<div class="main-content">
            <div class="hero"><div style="display:flex;align-items:center;gap:10px;margin-bottom:14px"><div style="width:36px;height:36px;border-radius:10px;background:rgba(196,30,58,.2);display:flex;align-items:center;justify-content:center">⚡</div><span style="color:#FF6B81;font-size:13px;font-weight:600">AI-Powered IT Support</span></div><h1 style="font-size:26px">שלום {sn}! 👋<br/><span style="font-size:20px;color:#6B7280">מערכת תמיכה חכמה</span></h1><p>סוכן AI חכם שפותר בעיות טכניות בזמן אמת, מציג מדריכים ויזואליים, ופותח קריאות שירות רק כשבאמת צריך.</p><div style="display:flex;gap:12px"><button class="btn-red" data-action="nav" data-value="chat">💬 יש לי בעיה טכנית</button><button class="btn-outline" data-action="nav" data-value="kb">עיין במדריכים</button></div></div>
            <h3 style="font-size:16px;font-weight:700;margin-bottom:14px">מה הבעיה? בחרו נושא:</h3>
            <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:28px">{th}</div>
            <h3 style="font-size:16px;font-weight:700;margin-bottom:14px">מדריכים פופולריים</h3>
            <div style="display:grid;grid-template-columns:repeat(2,1fr);gap:12px">{kh}</div>
        </div>"""

    def tickets_html():
        f_status = state.get('ticket_filter', 'all')
        tickets = fetch_tickets(filter_status=f_status if f_status != 'all' else None)
        bm = {"open":("פתוח","#FEF3C7","#92400E","#F59E0B"),"progress":("בטיפול","#DBEAFE","#1E40AF","#3B82F6"),"done":("נפתר","#D1FAE5","#065F46","#10B981"),"waiting":("ממתין","#FEE2E2","#991B1B","#EF4444")}
        pm = {"high":("גבוהה","#DC2626"),"med":("בינונית","#F59E0B"),"low":("נמוכה","#10B981"),"קריטית":("קריטית","#DC2626"),"דחופה":("דחופה","#F97316"),"בינונית":("בינונית","#F59E0B"),"נמוכה":("נמוכה","#10B981")}
        def fb(val, label):
            active = 'background:#1A1A2E;color:#fff;border-color:#1A1A2E' if f_status == val else ''
            return f'<button class="suggest-btn" data-action="filter-tickets" data-value="{val}" style="{active}">{label}</button>'
        filters = f'<div style="display:flex;gap:8px;margin-bottom:16px">{fb("all","הכל")}{fb("open","פתוח")}{fb("progress","בטיפול")}{fb("done","נפתר")}{fb("waiting","ממתין")}</div>'
        cards = ''
        if not tickets:
            cards = '<div style="padding:40px;text-align:center;color:#9CA3AF">📭 אין קריאות עדיין</div>'
        for t in tickets:
            st = t.get('status','open'); s = bm.get(st, bm['open']); p = pm.get(t.get('priority','med'), pm['med'])
            tid = t.get('ticket_number','')
            summary = t.get('ai_summary','') or t.get('last_update','') or ''
            created = str(t.get('created_at',''))[:16].replace('T',' ')
            # Status timeline - clickable buttons to update status
            status_map = [('open','פתוח'),('progress','בטיפול'),('done','נפתר')]
            timeline = '<div style="display:flex;align-items:center;gap:4px;margin-top:12px">'
            for si, (s_key, s_label) in enumerate(status_map):
                is_current = st == s_key
                if is_current:
                    timeline += f'<div style="padding:4px 10px;border-radius:6px;background:#059669;color:#fff;font-size:11px;font-weight:600;cursor:default">{s_label}</div>'
                else:
                    timeline += f'<div style="padding:4px 10px;border-radius:6px;background:#F3F4F6;color:#6B7280;font-size:11px;cursor:pointer;border:1px solid #E5E7EB" data-action="update-status" data-value="{tid}|{s_key}">{s_label}</div>'
                if si < len(status_map)-1:
                    timeline += f'<div style="width:20px;height:2px;background:#D1D5DB"></div>'
            timeline += '</div>'

            cards += f"""<div style="background:#fff;border-radius:14px;border:1px solid #E5E7EB;margin-bottom:12px;overflow:hidden">
                <div style="padding:18px 20px;display:flex;align-items:center;justify-content:space-between;cursor:pointer" data-action="toggle-ticket" data-value="{tid}">
                    <div style="display:flex;align-items:center;gap:14px">
                        <div style="font-size:13px;font-weight:700;color:#C41E3A;min-width:60px">{tid}</div>
                        <div><div style="font-size:14px;font-weight:600">{t.get('subject','')}</div>
                        <div style="font-size:12px;color:#9CA3AF">{t.get('student_name','')} · {t.get('category','')} · {created}</div></div>
                    </div>
                    <div style="display:flex;align-items:center;gap:10px">
                        <span style="font-weight:600;color:{p[1]};font-size:12px">{p[0]}</span>
                        <span class="badge" style="background:{s[1]};color:{s[2]}"><span style="width:6px;height:6px;border-radius:50%;background:{s[3]};display:inline-block"></span>{s[0]}</span>
                        <span style="color:#9CA3AF;font-size:16px">▼</span>
                    </div>
                </div>
                <div id="ticket-detail-{tid}" style="display:none;padding:0 20px 18px;border-top:1px solid #F3F4F6">
                    <div style="padding-top:14px;display:grid;grid-template-columns:80px 1fr;gap:8px 14px;font-size:13px">
                        <span style="color:#6B7280;font-weight:600">📝 תיאור</span><span>{summary}</span>
                        <span style="color:#6B7280;font-weight:600">📞 טלפון</span><span>{t.get('phone','לא צוין')}</span>
                        <span style="color:#6B7280;font-weight:600">📅 נפתח</span><span>{created}</span>
                        <span style="color:#6B7280;font-weight:600">📌 עדכון</span><span>{t.get('last_update','ממתין')}</span>
                    </div>
                    {timeline}
                </div>
            </div>"""
        return f"""<div class="main-content"><div style="margin-bottom:24px"><h2 style="font-size:22px;font-weight:800">קריאות שירות</h2><p style="font-size:13.5px;color:#6B7280">ניהול קריאות שנפתחו ע"י הסוכן</p></div>
            {filters}{cards}</div>"""

    def kb_html():
        cat_icons = {"סיסמאות":"🔑","Office":"📧","רשת":"📶","Moodle":"📚","הדפסה":"🖨️"}
        cat_colors = {"סיסמאות":("#FEF3C7","#92400E","#F59E0B"),"Office":("#DBEAFE","#1E40AF","#3B82F6"),"רשת":("#D1FAE5","#065F46","#10B981"),"Moodle":("#EDE9FE","#5B21B6","#8B5CF6"),"הדפסה":("#FCE7F3","#9D174D","#EC4899")}
        # Category tabs
        cats = list(set(k["cat"] for k in KB_ITEMS))
        tabs = '<div style="display:flex;gap:8px;margin-bottom:20px;flex-wrap:wrap">'
        tabs += '<div style="padding:8px 16px;border-radius:10px;background:#1A1A2E;color:#fff;font-size:13px;font-weight:600;cursor:default">הכל <span style="background:rgba(255,255,255,.2);padding:1px 6px;border-radius:6px;font-size:11px;margin-right:4px">{}</span></div>'.format(len(KB_ITEMS))
        for cat in cats:
            count = len([k for k in KB_ITEMS if k["cat"] == cat])
            cc = cat_colors.get(cat, ("#F3F4F6","#374151","#6B7280"))
            icon = cat_icons.get(cat, "📄")
            tabs += f'<div style="padding:8px 16px;border-radius:10px;background:{cc[0]};color:{cc[1]};font-size:13px;font-weight:500;cursor:default">{icon} {cat} <span style="font-size:11px;opacity:.6">{count}</span></div>'
        tabs += '</div>'
        # Cards
        kh = ''
        for k in KB_ITEMS:
            cc = cat_colors.get(k["cat"], ("#F3F4F6","#374151","#6B7280"))
            icon = cat_icons.get(k["cat"], "📄")
            tags = ''.join(f'<span style="padding:3px 10px;border-radius:6px;background:{cc[0]};font-size:11px;color:{cc[1]};font-weight:500">{t}</span>' for t in k["tags"])
            kh += f"""<a href="{k.get('url','#')}" target="_blank" style="text-decoration:none;color:inherit;display:block">
                <div class="kb-card">
                    <div style="position:absolute;top:0;right:0;width:80px;height:80px;background:{cc[0]};border-radius:0 0 0 80px;opacity:.5"></div>
                    <div style="display:flex;align-items:flex-start;gap:16px;position:relative">
                        <div style="width:52px;height:52px;border-radius:14px;background:linear-gradient(135deg,{cc[0]},{cc[0]}dd);display:flex;align-items:center;justify-content:center;font-size:26px;flex-shrink:0;box-shadow:0 2px 8px {cc[2]}30">{icon}</div>
                        <div style="flex:1">
                            <h4 style="font-size:15px;font-weight:700;margin-bottom:8px;color:#1A1A2E">{k['title']}</h4>
                            <div style="display:flex;gap:6px;margin-bottom:12px;flex-wrap:wrap">{tags}</div>
                            <div style="display:flex;justify-content:space-between;align-items:center">
                                <span style="font-size:12px;color:{cc[1]};font-weight:600;background:{cc[0]};padding:3px 10px;border-radius:6px">{k['cat']}</span>
                                <span style="font-size:12px;color:#9CA3AF;display:flex;align-items:center;gap:4px">👁 {k['views']:,}</span>
                            </div>
                        </div>
                    </div>
                    <div style="margin-top:14px;padding-top:12px;border-top:1px solid #F3F4F6;display:flex;align-items:center;justify-content:space-between">
                        <span style="font-size:12px;color:#6B7280">לחץ לצפייה במדריך המלא</span>
                        <span style="font-size:12px;color:#C41E3A;font-weight:600">צפה במדריך ↗</span>
                    </div>
                </div></a>"""
        return f"""<div class="main-content">
            <div style="margin-bottom:28px">
                <div style="display:flex;align-items:center;gap:12px;margin-bottom:8px"><div style="width:42px;height:42px;border-radius:12px;background:linear-gradient(135deg,#C41E3A,#E84563);display:flex;align-items:center;justify-content:center;font-size:22px;color:#fff">📚</div><h2 style="font-size:24px;font-weight:800">מאגר ידע</h2></div>
                <p style="font-size:14px;color:#6B7280">מדריכים, הנחיות ופתרונות למחלקת מחשוב</p>
            </div>
            {tabs}
            <div style="display:grid;grid-template-columns:repeat(2,1fr);gap:18px">{kh}</div>
        </div>"""

    def dashboard_html():
        sessions = fetch_chat_sessions(); tix = fetch_tickets(); activity = fetch_activity_log(5)
        ti = len(sessions); tt = len(tix)
        ai_pct = round((len([s for s in sessions if s.get('status')=='closed'])/max(ti,1))*100) if ti else 0
        saved = ti - tt
        avg_time = f'{sum(state["response_times"])/len(state["response_times"]):.1f}s' if state['response_times'] else '~3s'
        stats = '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:28px">'
        for label,val,emoji,color in [("סה\"כ פניות",f'{ti:,}','💬','#3B82F6'),('נפתרו ע"י AI',f'{ai_pct}%','🤖','#10B981'),('קריאות שנחסכו',f'{saved:,}','🎫','#C41E3A'),('זמן ממוצע',avg_time,'⏱️','#8B5CF6')]:
            stats += f'<div class="card"><div style="width:44px;height:44px;border-radius:12px;background:{color}12;display:flex;align-items:center;justify-content:center;font-size:22px;margin-bottom:16px">{emoji}</div><div style="font-size:28px;font-weight:800;color:#1A1A2E;margin-bottom:4px">{val}</div><div style="font-size:13px;color:#6B7280">{label}</div></div>'
        stats += '</div>'
        act = ''
        em = {"red":"🔴","blue":"🔵","green":"🟢","purple":"📄"}; bc = {"red":"#FEF2F2","blue":"#EFF6FF","green":"#ECFDF5","purple":"#F5F3FF"}
        if not activity: act = '<div style="color:#9CA3AF;font-size:13px">אין פעילות עדיין</div>'
        for i, a in enumerate(activity):
            it = a.get("icon_type","blue"); bd = 'border-bottom:1px solid #F3F4F6;' if i < len(activity)-1 else ''
            act += f'<div style="display:flex;align-items:center;gap:12px;padding:10px 0;{bd}"><div style="width:34px;height:34px;border-radius:8px;background:{bc.get(it,"#EFF6FF")};display:flex;align-items:center;justify-content:center;font-size:15px;flex-shrink:0">{em.get(it,"🔵")}</div><div style="flex:1"><div style="font-size:13px;font-weight:500">{a.get("text","")}</div><div style="font-size:11px;color:#9CA3AF">{str(a.get("created_at",""))[:16]}</div></div></div>'
        return f'<div class="main-content"><div style="margin-bottom:28px"><h2 style="font-size:24px;font-weight:800">דשבורד מנהל</h2><p style="font-size:14px;color:#6B7280">סטטיסטיקות ותובנות</p></div>{stats}<div class="card"><h3 style="font-size:15px;font-weight:700;margin-bottom:16px">פעילות אחרונה</h3>{act}</div></div>'

    def my_tickets_html():
        tickets = fetch_tickets(student_name=state['student_name'])
        bm = {"open":("פתוח","#FEF3C7","#92400E","#F59E0B"),"progress":("בטיפול","#DBEAFE","#1E40AF","#3B82F6"),"done":("נפתר","#D1FAE5","#065F46","#10B981"),"waiting":("ממתין","#FEE2E2","#991B1B","#EF4444")}
        si = {"open":("#FEF3C7","#D97706","🕐"),"progress":("#DBEAFE","#2563EB","🔧"),"done":("#D1FAE5","#059669","✅"),"waiting":("#FEE2E2","#DC2626","⏳")}
        cards = ''
        if not tickets:
            cards = '<div style="text-align:center;padding:60px;color:#9CA3AF"><div style="font-size:40px;margin-bottom:12px">🎉</div><div style="font-size:16px;font-weight:600">אין קריאות פתוחות!</div></div>'
        for t in tickets:
            st = t.get('status','open'); s = si.get(st,si['open']); b = bm.get(st,bm['open'])
            cards += f'<div style="background:#fff;border-radius:14px;padding:20px;border:1px solid #E5E7EB;display:flex;align-items:center;justify-content:space-between;margin-bottom:12px"><div style="display:flex;align-items:center;gap:14px"><div style="width:42px;height:42px;border-radius:10px;background:{s[0]};display:flex;align-items:center;justify-content:center;font-size:20px">{s[2]}</div><div><div style="font-size:14px;font-weight:600">{t.get("subject","")}</div><div style="font-size:12px;color:#9CA3AF">{t.get("ticket_number","")} · {t.get("category","")}</div></div></div><span class="badge" style="background:{b[1]};color:{b[2]}"><span style="width:6px;height:6px;border-radius:50%;background:{b[3]};display:inline-block"></span>{b[0]}</span></div>'
        return f'<div class="main-content"><div style="margin-bottom:24px"><h2 style="font-size:22px;font-weight:800">הקריאות שלי</h2><p style="font-size:13.5px;color:#6B7280">מעקב אחרי קריאות שירות</p></div>{cards}</div>'

    # ===== RENDER =====
    def login_html():
        return """<div style="width:100vw;height:100vh;display:flex;align-items:center;justify-content:center;background:linear-gradient(135deg,#1A1A2E,#16213E,#0F3460);direction:rtl">
            <div style="background:#fff;border-radius:20px;padding:48px 40px;width:400px;max-width:90vw;box-shadow:0 20px 60px rgba(0,0,0,.3);text-align:center">
                <div style="width:64px;height:64px;border-radius:16px;background:linear-gradient(135deg,#C41E3A,#E84563);display:flex;align-items:center;justify-content:center;margin:0 auto 20px;box-shadow:0 4px 16px rgba(196,30,58,.3)"><span style="font-size:30px;color:#fff">🤖</span></div>
                <h1 style="font-size:24px;font-weight:800;color:#1A1A2E;margin-bottom:6px">מערכת תמיכה חכמה</h1>
                <p style="font-size:14px;color:#6B7280;margin-bottom:28px">עזריאלי - מכללה אקדמית להנדסה ירושלים</p>
                <input id="loginName" type="text" placeholder="שם מלא" style="width:100%;padding:14px 16px;border-radius:12px;border:1px solid #E5E7EB;font-size:14px;font-family:inherit;margin-bottom:12px;direction:rtl;outline:none;text-align:right"/>
                <input id="loginPhone" type="tel" placeholder="מספר טלפון" style="width:100%;padding:14px 16px;border-radius:12px;border:1px solid #E5E7EB;font-size:14px;font-family:inherit;margin-bottom:20px;direction:ltr;outline:none;text-align:right"/>
                <button style="width:100%;padding:14px;border-radius:12px;border:none;background:linear-gradient(135deg,#C41E3A,#E84563);color:#fff;font-size:15px;font-weight:700;cursor:pointer;box-shadow:0 4px 16px rgba(196,30,58,.4);font-family:inherit" data-action="login" data-value="1">כניסה למערכת ←</button>
                <p style="font-size:11px;color:#9CA3AF;margin-top:16px">הסוכן החכם שלנו יכול לפתור את רוב הבעיות תוך שניות</p>
            </div>
        </div>"""

    def render_all():
        if not state['logged_in']:
            app_container.set_content(login_html())
            return
        sb = sidebar_html()
        pg = get_page_html()
        app_container.set_content(sb + pg)

    def get_page_html():
        p, r = state['page'], state['role']
        if p == 'home' and r == 'admin': return home_admin_html()
        elif p == 'home': return home_student_html()
        elif p == 'chat':
            if not state['chat_messages']:
                g = TOPIC_GREETINGS.get(state.get('chat_topic'), "שלום! 👋 אני הסוכן החכם של מחלקת המחשוב.\nאיך אוכל לעזור?")
                state['chat_messages'] = [{'role': 'bot', 'content': g}]
                if not state.get('session_created'):
                    try:
                        supabase_client.table("chat_sessions").insert({"thread_id": state['thread_id'], "student_name": state['student_name'], "phone": state.get('student_phone', ''), "topic": state.get('chat_topic') or 'other', "status": "active"}).execute()
                        state['session_created'] = True
                    except: pass
            return chat_html()
        elif p == 'tickets' and r == 'admin': return tickets_html()
        elif p == 'kb': return kb_html()
        elif p == 'my-tickets': return my_tickets_html()
        else: return home_student_html()

    # ===== HANDLERS =====
    def show_toast(msg, color='#1A1A2E'):
        try:
            ui.run_javascript(f"""(function(){{var t=document.createElement('div');t.className='toast';t.style.background='{color}';t.style.color='#fff';t.textContent='{msg}';document.body.appendChild(t);setTimeout(function(){{t.remove()}},3000)}})()""")
        except: pass

    def handle_navigate(pid):
        state['page'] = pid
        render_all()

    def handle_role(r):
        # Save current role's chat
        old_role = state['role']
        state['saved_chats'][old_role] = {
            'chat_messages': state['chat_messages'],
            'chat_topic': state['chat_topic'],
            'thread_id': state['thread_id'],
            'session_created': state['session_created'],
            'pending_ticket': state['pending_ticket'],
            'pending_tc_id': state['pending_tc_id'],
        }
        # Restore new role's chat (or start fresh)
        saved = state['saved_chats'].get(r)
        if saved:
            state['chat_messages'] = saved['chat_messages']
            state['chat_topic'] = saved['chat_topic']
            state['thread_id'] = saved['thread_id']
            state['session_created'] = saved['session_created']
            state['pending_ticket'] = saved['pending_ticket']
            state['pending_tc_id'] = saved['pending_tc_id']
        else:
            state['chat_messages'] = []; state['chat_topic'] = None
            state['thread_id'] = str(uuid4()); state['session_created'] = False
            state['pending_ticket'] = None; state['pending_tc_id'] = None
        state['role'] = r; state['page'] = 'home'
        state['is_admin'] = (r == 'admin')
        render_all()

    def handle_topic(t):
        state['chat_topic'] = t; state['chat_messages'] = []
        state['page'] = 'chat'; state['thread_id'] = str(uuid4()); state['session_created'] = True
        try:
            supabase_client.table("chat_sessions").insert({"thread_id": state['thread_id'], "student_name": state['student_name'], "phone": state.get('student_phone', ''), "topic": t, "status": "active"}).execute()
        except Exception as e:
            print(f"chat_session insert error: {e}")
        render_all()

    def handle_new_chat():
        state['chat_messages'] = []; state['chat_topic'] = None
        state['thread_id'] = str(uuid4()); state['session_created'] = False
        state['pending_ticket'] = None; state['pending_tc_id'] = None
        render_all()

    async def handle_send_msg(msg_text):
        if not msg_text or not msg_text.strip(): return
        import time as _time

        # Handle shortcut suggestion buttons
        if 'חזרה לדף הבית' in msg_text:
            handle_navigate('home'); return
        if 'שאלה חדשה' in msg_text:
            handle_new_chat(); return

        # HITL resume
        if state.get('pending_ticket'):
            ticket = state['pending_ticket']
            tc_id = state.get('pending_tc_id', '')
            if any(w in msg_text for w in ['מאשר','אשר','כן']):
                state['chat_messages'].append({'role': 'user', 'content': msg_text})
                state['chat_messages'].append({'role': 'bot', 'content': '...'})
                state['busy'] = True
                try:
                    cfg = {"configurable": {"thread_id": state['thread_id']}, "recursion_limit": 10}
                    tool_msg = ToolMessage(tool_call_id=tc_id, content="Ticket approved.", name="ProposeTicket")
                    await asyncio.get_event_loop().run_in_executor(None, lambda: graph_instance.invoke({"messages": [tool_msg]}, cfg))
                    try:
                        t_num = f"T-{str(uuid4())[:4].upper()}"
                        supabase_client.table("tickets").insert({"ticket_number": t_num, "subject": ticket.get('subject',''), "category": ticket.get('category','כללי'), "priority": ticket.get('priority','med'), "status": "open", "student_name": state['student_name'], "phone": state.get('student_phone', ''), "ai_summary": ticket.get('description',''), "confidence_score": 2, "message_count": len([m for m in state['chat_messages'] if m['role']=='user']), "last_update": "ממתין לטיפול טכנאי"}).execute()
                        supabase_client.table("activity_log").insert({"text": f"קריאה {t_num} נפתחה - {ticket.get('subject','')}", "icon_type": "red"}).execute()
                        try: supabase_client.table("chat_sessions").update({"status": "escalated"}).eq("thread_id", state['thread_id']).execute()
                        except: pass
                        state['chat_messages'][-1] = {'role': 'bot', 'content': f"✅ קריאת שירות {t_num} נפתחה בהצלחה!\n\n📋 נושא: {ticket.get('subject','')}\n📁 קטגוריה: {ticket.get('category','')}\n🔺 עדיפות: {ticket.get('priority','')}\n\nהטכנאי יצור קשר בהקדם.", 'suggestions': ["חזרה לדף הבית 🏠"]}
                        show_toast('✅ קריאה נפתחה בהצלחה', '#059669')
                    except Exception as db_err:
                        state['chat_messages'][-1] = {'role': 'bot', 'content': f"✅ הקריאה אושרה אך שגיאה בשמירה: {str(db_err)[:80]}"}
                except Exception as ex:
                    state['chat_messages'][-1] = {'role': 'bot', 'content': f"✅ קריאה אושרה!\n\n📋 {ticket.get('subject','')}\n\nהטכנאי יצור קשר בהקדם."}
                state['pending_ticket'] = None; state['pending_tc_id'] = None
            else:
                state['chat_messages'].append({'role': 'user', 'content': msg_text})
                state['pending_ticket'] = None; state['pending_tc_id'] = None
                state['thread_id'] = str(uuid4())
                state['chat_messages'].append({'role': 'bot', 'content': "בסדר, לא נפתחה קריאה. בוא ננסה לפתור את זה ביחד.\nספר לי יותר על הבעיה - מה בדיוק קורה?", 'suggestions': ["תסביר יותר 📝", "שאלה חדשה 💬"]})
            state['busy'] = False; state['needs_render'] = True; return

        # Normal message - run agent in BACKGROUND THREAD
        state['chat_messages'].append({'role': 'user', 'content': msg_text})
        state['chat_messages'].append({'role': 'bot', 'content': '...'})
        try:
            safe_text = msg_text.replace("'", "\\'").replace('"', '\\"').replace('\n', '<br>')
            await ui.run_javascript(f"""(function(){{var cm=document.getElementById('chatMessages');if(!cm)return;
                cm.innerHTML+='<div class="msg-row user"><div class="msg-avatar user">👤</div><div class="msg-bubble user">{safe_text}</div></div>'
                +'<div class="msg-row" id="thinkingRow"><div class="msg-avatar bot">🤖</div><div class="msg-bubble bot" style="min-width:300px">'
                +'<div style="display:flex;align-items:center;gap:10px;margin-bottom:10px"><span class="typing-dot"></span><span class="typing-dot"></span><span class="typing-dot"></span><span style="font-size:14px;color:#1A1A2E;font-weight:600">מעבד את הבקשה...</span></div>'
                +'<div id="thinkToggle" style="display:inline-flex;align-items:center;gap:6px;padding:6px 14px;background:#F3F4F6;border-radius:8px;cursor:pointer;font-size:12px;color:#6B7280;border:1px solid #E5E7EB" onclick="var s=document.getElementById(\\'thinkSteps\\');if(s)s.style.display=s.style.display===\\'none\\'?\\'block\\':\\'none\\';this.querySelector(\\'span\\').textContent=s.style.display===\\'none\\'?\\'▶\\':&apos;▼&apos;"><span>▶</span>🔍 הצג תהליך חשיבה</div>'
                +'<div id="thinkSteps" style="display:none;padding:10px 0;font-size:12px;line-height:2;border-top:1px solid #F3F4F6;margin-top:8px">'
                +'<div style="color:#3B82F6">🔍 מחפש במאגר הידע של המכללה...</div>'
                +'</div>'
                +'</div></div>';
                cm.scrollTop=cm.scrollHeight;
                window._thinkTimer=setInterval(function(){{
                    var el=document.getElementById("thinkSteps");
                    if(!el){{clearInterval(window._thinkTimer);return}}
                    var steps=el.children.length;
                    if(steps===1)el.innerHTML+="<div style=\\"color:#10B981\\">📚 בודק מדריכים רלוונטיים במאגר הידע...</div>";
                    else if(steps===2)el.innerHTML+="<div style=\\"color:#8B5CF6\\">🤔 מנתח את המידע שנמצא...</div>";
                    else if(steps===3)el.innerHTML+="<div style=\\"color:#F59E0B\\">✍️ מנסח תשובה מותאמת עם מקורות...</div>";
                    else if(steps===4)el.innerHTML+="<div style=\\"color:#EF4444\\">🌐 מחפש מידע נוסף באינטרנט...</div>";
                    else clearInterval(window._thinkTimer);
                }},2500);
            }})()""")
        except: pass
        import threading
        def _bg():
            import time as _t2, re as _re2
            t0=_t2.time()
            try:
                cfg={"configurable":{"thread_id":state['thread_id']},"recursion_limit":8}
                result=graph_instance.invoke({"messages":[HumanMessage(content=msg_text)],"is_admin":state['is_admin'],"awaiting_hitl":False,"ticket_draft":{},"request_type":"","complexity":"","confidence_score":0,"retry_count":0},cfg)
                rt=_t2.time()-t0; state['response_times'].append(rt)

                # Analyze which tools were used - ONLY from current cycle
                # Find last HumanMessage index to ignore old tool results
                tools_used = []
                web_sources = []
                guide_sources = []
                last_human_idx = -1
                all_msgs = result["messages"]
                for i, m in enumerate(all_msgs):
                    if isinstance(m, HumanMessage):
                        last_human_idx = i
                # Only look at messages AFTER the last HumanMessage
                current_msgs = all_msgs[last_human_idx+1:] if last_human_idx >= 0 else all_msgs
                print(f"📊 Total msgs: {len(all_msgs)}, Current cycle: {len(current_msgs)}")
                for m in current_msgs:
                    if isinstance(m, AIMessage) and hasattr(m, 'tool_calls') and m.tool_calls:
                        for tc in m.tool_calls:
                            tools_used.append(tc["name"])
                    if isinstance(m, ToolMessage) and m.content:
                        content_str = m.content
                        # Check if this is a retrieve result (has Guide: marker) or tavily result
                        is_kb = 'Guide:' in content_str and 'Source:' in content_str
                        if is_kb:
                            # KB guide sources
                            for block in content_str.split('---'):
                                gm = _re2.search(r'Guide:\s*(.+?)\s*\|', block)
                                um = _re2.search(r'URL:\s*(https?://[^\s|]+)', block)
                                if gm and um:
                                    guide_sources.append({'name': gm.group(1).strip(), 'url': um.group(1).strip()})
                        else:
                            # Tavily web sources
                            for block in content_str.split('---'):
                                um = _re2.search(r'URL:\s*(https?://[^\s]+)', block)
                                tm = _re2.search(r'Title:\s*(.+?)(?:\n|$)', block)
                                if um:
                                    web_sources.append({
                                        'url': um.group(1).strip(),
                                        'title': tm.group(1).strip() if tm else um.group(1).strip()
                                    })

                print(f"🔧 Tools used: {tools_used}")
                if web_sources: print(f"🌐 Web sources: {[s['title'][:30] for s in web_sources]}")
                if guide_sources: print(f"📚 Guide sources: {[s['name'][:30] for s in guide_sources]}")

                ai=""; hitl=False
                for m in reversed(result["messages"]):
                    if isinstance(m, AIMessage):
                        if hasattr(m,'tool_calls') and m.tool_calls:
                            for tc in m.tool_calls:
                                if tc["name"]=="ProposeTicket":
                                    state['pending_ticket']=tc["args"]; state['pending_tc_id']=tc["id"]
                                    state['chat_messages'].pop()
                                    state['chat_messages'].append({'role':'bot','content':'','is_hitl':True,'ticket_data':tc["args"]})
                                    hitl=True; break
                        if not hitl and m.content: ai=m.content
                        break
                if not hitl:
                    gi=[]; isrc=None
                    try:
                        sf=None; guide_name_clean=None; guide_url=None
                        # Collect ALL guides from current cycle tool results only
                        all_guides = []
                        for m in current_msgs:
                            if isinstance(m,ToolMessage) and m.content:
                                for block in m.content.split('---'):
                                    sm=_re2.search(r'Source:\s*(.+?\.pdf)',block)
                                    if sm:
                                        nm=_re2.search(r'Guide:\s*(.+?)\s*\|',block)
                                        um=_re2.search(r'URL:\s*(https?://[^\s|]+)',block)
                                        all_guides.append({
                                            'sf': sm.group(1),
                                            'name': nm.group(1).strip() if nm else sm.group(1).replace('.pdf',''),
                                            'url': um.group(1).strip() if um else None
                                        })
                        print(f"📚 Found {len(all_guides)} guides: {[g['name'][:30] for g in all_guides]}")

                        # Match: find which guide the AI actually used in its response
                        best_guide = None
                        if all_guides and ai:
                            # Method 1: Match by specific URL (/KB/View/XXX) in AI response
                            for g in all_guides:
                                if g['url'] and '/KB/View/' in (g['url'] or '') and g['url'] in ai:
                                    best_guide = g
                                    print(f"✅ Matched by URL: '{g['name']}' ({g['url']})")
                                    break
                            # Method 2: Match by guide name - check if AI response contains the guide name
                            if not best_guide:
                                for g in all_guides:
                                    # Check if a significant portion of the guide name appears in AI text
                                    g_words = [w for w in g['name'].lower().split() if len(w) > 2]
                                    ai_lower = ai.lower()
                                    if len(g_words) > 0:
                                        matches = sum(1 for w in g_words if w in ai_lower)
                                        ratio = matches / len(g_words)
                                        if ratio >= 0.5 and matches >= 2:
                                            best_guide = g
                                            print(f"✅ Matched by name: '{g['name']}' ({matches}/{len(g_words)} words, {ratio:.0%})")
                                            break
                            # NO FALLBACK - if we can't confidently match, don't show images
                            if not best_guide:
                                print(f"⚠️ Could not match any guide to AI response - no images")
                        if best_guide:
                            sf = best_guide['sf']
                            guide_name_clean = best_guide['name']
                            guide_url = best_guide['url']
                            isrc = guide_name_clean
                            idir='/content/kb_images/'
                            if os.path.exists(idir):
                                for f in sorted(os.listdir(idir)):
                                    if f.startswith(sf) and f.endswith('.png'): gi.append(os.path.join(idir,f))
                            print(f"🖼️ Showing {len(gi)} pages from: '{guide_name_clean}'")
                    except Exception as ie: print(f"⚠️ Img err: {ie}")
                    if ai:
                        ai=_re2.sub(r'!\[[^\]]*\]\(sandbox:[^)]+\)','',ai).strip()
                        ai=_re2.sub(r'\(sandbox:[^)]+\)','',ai).strip()
                        ai=_re2.sub(r'\n{3,}','\n\n',ai).strip()
                    # Build tools/sources metadata
                    tools_meta = {'tools_used': tools_used, 'web_sources': web_sources, 'guide_sources': guide_sources}
                    state['chat_messages'][-1]={'role':'bot','content':ai or "לא הצלחתי לעבד.",'guide_images':gi,'image_source':isrc,'resp_time':rt,'tools_meta':tools_meta}
            except Exception as ex:
                print(f"❌ Agent error: {ex}")
                state['chat_messages'][-1]={'role':'bot','content':f"שגיאה: {str(ex)[:100]}"}
            state['needs_render']=True
            print(f"✅ Agent thread done. needs_render=True. Messages: {len(state['chat_messages'])}")
        threading.Thread(target=_bg, daemon=True).start()
        print("🚀 Agent thread started")

    async def handle_image_upload():
        try:
            # First check if data exists (small call)
            has_data = await ui.run_javascript("document.getElementById('imgData')?.value?.length > 100 ? 'yes' : 'no'")
            if has_data != 'yes':
                print("⚠️ No image data found")
                return
            # Read the resized image data
            img_data = await ui.run_javascript("(function(){var e=document.getElementById('imgData');var v=e?e.value:'';e.value='';return v})()")
            if not img_data or not img_data.startswith('data:image'):
                print(f"⚠️ Invalid image data: {str(img_data)[:50]}")
                return
            print(f"📷 Image received: {len(img_data)} chars")
            state['chat_messages'].append({'role':'user','content':'📷 שלחתי תמונה','user_image':img_data})
            state['chat_messages'].append({'role':'bot','content':'...'})
            # Inject typing via JS
            try:
                await ui.run_javascript("""(function(){var cm=document.getElementById('chatMessages');if(!cm)return;
                    cm.innerHTML+='<div class="msg-row user"><div class="msg-avatar user">👤</div><div class="msg-bubble user">📷 שלחתי תמונה</div></div>'
                    +'<div class="msg-row"><div class="msg-avatar bot">🤖</div><div class="msg-bubble bot"><span class="typing-dot"></span><span class="typing-dot"></span><span class="typing-dot"></span> מנתח תמונה...</div></div>';
                    cm.scrollTop=cm.scrollHeight})()""")
            except: pass
            # Run in background thread
            import threading
            _img = img_data
            def _bg_img():
                try:
                    cfg = {"configurable": {"thread_id": state['thread_id']}, "recursion_limit": 10}
                    vision_msg = HumanMessage(content=[
                        {"type":"text","text":"המשתמש שלח צילום מסך של בעיה טכנית. נתח את התמונה ועזור לו. ענה בעברית."},
                        {"type":"image_url","image_url":{"url":_img}}])
                    result = graph_instance.invoke({"messages":[vision_msg],"is_admin":state['is_admin'],"awaiting_hitl":False,"ticket_draft":{},"request_type":"","complexity":"","confidence_score":0,"retry_count":0},cfg)
                    ai = ""
                    for m in reversed(result["messages"]):
                        if isinstance(m, AIMessage) and m.content: ai = m.content; break
                    state['chat_messages'][-1] = {'role':'bot','content': ai or "לא הצלחתי לנתח את התמונה."}
                    print(f"✅ Vision done: {ai[:60]}")
                except Exception as ex:
                    print(f"❌ Vision error: {ex}")
                    state['chat_messages'][-1] = {'role':'bot','content': "לא הצלחתי לנתח את התמונה. נסה לתאר את הבעיה במילים."}
                state['needs_render'] = True
            threading.Thread(target=_bg_img, daemon=True).start()
            print("🚀 Vision thread started")
        except Exception as ex:
            print(f"❌ Image upload error: {ex}")
            # Don't crash - show message
            state['chat_messages'].append({'role':'bot','content': "לא הצלחתי לקבל את התמונה. נסה לתאר את הבעיה במילים."})
            state['needs_render'] = True

    async def handle_rate(data):
        direction, idx = data.split('-', 1)
        try:
            supabase_client.table("activity_log").insert({"text": f"דירוג {'👍' if direction=='up' else '👎'} על תשובה", "icon_type": "green" if direction=='up' else "purple"}).execute()
            show_toast('תודה על המשוב! 🙏', '#059669' if direction=='up' else '#6B7280')
        except: pass

    async def handle_resolved():
        """User confirmed the issue was resolved - close the chat session"""
        try:
            supabase_client.table("chat_sessions").update({"status": "closed"}).eq("thread_id", state['thread_id']).execute()
            supabase_client.table("activity_log").insert({"text": f"בעיה נפתרה ע\"י AI - {state.get('student_name','')}", "icon_type": "green"}).execute()
        except Exception as e:
            print(f"Resolved update error: {e}")
        state['chat_messages'].append({'role': 'bot', 'content': '🎉 שמח שהצלחתי לעזור! אם תצטרך עזרה נוספת, אני תמיד כאן.', 'suggestions': ["שאלה חדשה 💬", "חזרה לדף הבית 🏠"]})
        show_toast('✅ הבעיה סומנה כנפתרה!', '#059669')
        render_all()

    async def handle_login():
        try:
            name = await ui.run_javascript("document.getElementById('loginName')?.value || ''")
            phone = await ui.run_javascript("document.getElementById('loginPhone')?.value || ''")
            if not name or not name.strip():
                show_toast('⚠️ נא להזין שם מלא', '#DC2626')
                return
            state['student_name'] = name.strip()
            state['student_phone'] = phone.strip() if phone else ''
            state['logged_in'] = True
            print(f"📱 Login: name={name.strip()}, phone={state['student_phone']}")
            # Save to persistent storage
            app.storage.user['logged_in'] = True
            app.storage.user['student_name'] = name.strip()
            app.storage.user['student_phone'] = phone.strip() if phone else ''
            show_toast(f'!שלום {name.strip()} 👋', '#059669')
            render_all()
        except Exception as e:
            print(f"Login error: {e}")

    # ===== JAVASCRIPT BRIDGE =====
    ui.add_body_html("""<input type="hidden" id="hv" value="">
    <textarea id="imgData" style="display:none"></textarea>
    <div id="lightbox" style="display:none;position:fixed;top:0;left:0;right:0;bottom:0;background:rgba(0,0,0,.9);z-index:9999;align-items:center;justify-content:center" data-action="closelb" data-value="1">
        <img id="lbImg" style="max-width:88%;max-height:88%;border-radius:8px;box-shadow:0 4px 30px rgba(0,0,0,.5)"/>
        <div style="position:absolute;top:16px;left:16px;color:#fff;font-size:28px;cursor:pointer;background:rgba(0,0,0,.5);width:44px;height:44px;border-radius:50%;display:flex;align-items:center;justify-content:center" data-action="closelb" data-value="1">✕</div>
        <div id="lbCounter" style="position:absolute;top:16px;left:50%;transform:translateX(-50%);color:#fff;font-size:14px;background:rgba(0,0,0,.5);padding:6px 16px;border-radius:20px"></div>
        <div id="lbPrev" style="display:none;position:absolute;right:20px;top:50%;transform:translateY(-50%);color:#fff;font-size:36px;cursor:pointer;background:rgba(0,0,0,.5);width:50px;height:50px;border-radius:50%;align-items:center;justify-content:center" data-action="lb-prev" data-value="1">→</div>
        <div id="lbNext" style="display:none;position:absolute;left:20px;top:50%;transform:translateY(-50%);color:#fff;font-size:36px;cursor:pointer;background:rgba(0,0,0,.5);width:50px;height:50px;border-radius:50%;align-items:center;justify-content:center" data-action="lb-next" data-value="1">←</div>
    </div>
    <script>
    window.carousels=window.carousels||{};
    function pyCall(a,d){document.getElementById("hv").value=a+"|"+d}
    function sendChat(){var i=document.getElementById("chatInput");if(i&&i.value.trim()){pyCall("send",i.value.trim());i.value=""}}
    function handleImageUpload(input){
        if(!input.files||!input.files[0])return;
        var file=input.files[0],reader=new FileReader();
        reader.onload=function(e){
            // Resize image to max 800px to avoid huge base64
            var img=new Image();
            img.onload=function(){
                var c=document.createElement("canvas");
                var max=800;
                var w=img.width,h=img.height;
                if(w>max||h>max){if(w>h){h=Math.round(h*max/w);w=max}else{w=Math.round(w*max/h);h=max}}
                c.width=w;c.height=h;
                c.getContext("2d").drawImage(img,0,0,w,h);
                document.getElementById("imgData").value=c.toDataURL("image/jpeg",0.7);
                pyCall("image","uploaded");
            };
            img.src=e.target.result;
        };
        reader.readAsDataURL(file);input.value="";
    }
    window._lbUid=null;
    function showLightbox(src,uid){
        var lb=document.getElementById("lightbox");
        document.getElementById("lbImg").src=src;
        lb.style.display="flex";
        window._lbUid=uid||null;
        var prev=document.getElementById("lbPrev"),next=document.getElementById("lbNext"),ctr=document.getElementById("lbCounter");
        if(uid&&window.carousels[uid]&&window.carousels[uid].imgs.length>1){
            prev.style.display="flex";next.style.display="flex";
            var c=window.carousels[uid];
            ctr.innerText="עמוד "+(c.idx+1)+" מתוך "+c.imgs.length;
        }else{prev.style.display="none";next.style.display="none";ctr.innerText="";}
    }
    function lbNav(dir){
        var uid=window._lbUid;if(!uid)return;
        var c=window.carousels[uid];if(!c)return;
        c.idx+=dir;if(c.idx<0)c.idx=c.imgs.length-1;if(c.idx>=c.imgs.length)c.idx=0;
        document.getElementById("lbImg").src=c.imgs[c.idx];
        document.getElementById("lbCounter").innerText="עמוד "+(c.idx+1)+" מתוך "+c.imgs.length;
        var img=document.getElementById("carousel_"+uid),counter=document.getElementById("counter_"+uid);
        if(img)img.src=c.imgs[c.idx];if(counter)counter.innerText="עמוד "+(c.idx+1)+" מתוך "+c.imgs.length;
    }
    function closeLightbox(){document.getElementById("lightbox").style.display="none";window._lbUid=null}
    function carouselNav(uid,dir){
        var c=window.carousels&&window.carousels[uid];if(!c)return;
        c.idx=c.idx+dir;if(c.idx<0)c.idx=c.imgs.length-1;if(c.idx>=c.imgs.length)c.idx=0;
        var img=document.getElementById("carousel_"+uid),counter=document.getElementById("counter_"+uid);
        if(img)img.src=c.imgs[c.idx];if(counter)counter.innerText="עמוד "+(c.idx+1)+" מתוך "+c.imgs.length;
    }
    document.addEventListener("click",function(e){
        var t=e.target.closest("[data-action]");if(!t)return;
        var a=t.dataset.action,v=t.dataset.value;
        if(a==="sendchat")sendChat();
        else if(a==="faq"){var inp=document.getElementById("chatInput");if(inp)inp.value=v;}
        else if(a==="camera"){var c=document.getElementById("camInput");if(c)c.click()}
        else if(a==="upload"){var u=document.getElementById("imgInput");if(u)u.click()}
        else if(a==="zoom"){var car=window.carousels&&window.carousels[v];if(car)showLightbox(car.imgs[car.idx],v)}
        else if(a==="zoom-single")showLightbox(v,null);
        else if(a==="toggle-ticket"){var d=document.getElementById("ticket-detail-"+v);if(d)d.style.display=d.style.display==="none"?"block":"none"}
        else if(a==="edit-ticket"){pyCall("edit-ticket","1")}
        else if(a==="submit-edit"){pyCall("submit-edit","1")}
        else if(a==="update-status"){pyCall("update-status",v)}
        else if(a==="copy-code"){var cb=e.target.closest('[data-action="copy-code"]');var card=cb.closest('[data-cmd]');if(card){var txt=card.getAttribute('data-cmd');navigator.clipboard.writeText(txt).then(function(){cb.innerHTML='✅ הועתק!';setTimeout(function(){cb.innerHTML='📋 העתק'},2000)}).catch(function(){var ta=document.createElement('textarea');ta.value=txt;document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);cb.innerHTML='✅ הועתק!';setTimeout(function(){cb.innerHTML='📋 העתק'},2000)})}}
        else if(a==="closelb")closeLightbox();
        else if(a==="lb-prev")lbNav(-1);
        else if(a==="lb-next")lbNav(1);
        else if(a==="carousel-prev")carouselNav(v,-1);
        else if(a==="carousel-next")carouselNav(v,1);
        else pyCall(a,v);
    });
    document.addEventListener("keydown",function(e){
        if(e.key==="Enter"&&e.target.id==="chatInput")sendChat();
        if(e.key==="Enter"&&(e.target.id==="loginName"||e.target.id==="loginPhone"))pyCall("login","1");
        if(e.key==="Escape")closeLightbox();
        if(e.key==="ArrowLeft"&&window._lbUid)lbNav(1);
        if(e.key==="ArrowRight"&&window._lbUid)lbNav(-1);
    });
    document.addEventListener("change",function(e){
        if(e.target.id==="camInput"||e.target.id==="imgInput")handleImageUpload(e.target);
    });
    var obs=new MutationObserver(function(){
        var el=document.getElementById("chatMessages");
        if(el){el.scrollTop=el.scrollHeight;
            // Force all links inside chat to open in new tab
            el.querySelectorAll("a[href]").forEach(function(a){a.setAttribute("target","_blank");a.setAttribute("rel","noopener")});
        }
        // Also fix links in main content area
        var mc=document.querySelector(".main-content");
        if(mc)mc.querySelectorAll("a[href]").forEach(function(a){if(a.href.startsWith("http"))a.setAttribute("target","_blank")});
    });
    setInterval(function(){
        var el=document.getElementById("chatMessages");
        if(el&&!el._observed){obs.observe(el,{childList:true,subtree:true});el._observed=true}
        // Init carousels from data attributes
        document.querySelectorAll("[data-carousel-id]").forEach(function(d){
            var uid=d.dataset.carouselId;
            if(!window.carousels[uid]){
                var imgs=d.dataset.carouselImgs.split(",");
                window.carousels[uid]={imgs:imgs,idx:0};
            }
        });
    },500);
    </script>""")

    async def check_bridge():
        # ALWAYS run javascript to keep WebSocket alive (heartbeat)
        try:
            val = await ui.run_javascript("(function(){var e=document.getElementById('hv');var v=e?e.value:'';if(v)e.value='';return v})()")
        except:
            return
        # Deferred render after invoke completes
        if state.get('needs_render'):
            state['needs_render'] = False
            print("🔄 Timer: rendering response...")
            try:
                await ui.run_javascript("if(window._thinkTimer)clearInterval(window._thinkTimer)")
                render_all()
                # Scroll chat to bottom after render
                await ui.run_javascript("var el=document.getElementById('chatMessages');if(el)el.scrollTop=el.scrollHeight")
                print("✅ Timer: render complete")
            except Exception as render_err:
                print(f"❌ Timer render error: {render_err}")
            return
        # Skip processing new actions while agent is working
        if state.get('busy'): return
        # Process action
        if val and '|' in val:
            action, data = val.split('|', 1)
            if action == 'nav': handle_navigate(data)
            elif action == 'role': handle_role(data)
            elif action == 'topic': handle_topic(data)
            elif action == 'send': await handle_send_msg(data)
            elif action == 'image': await handle_image_upload()
            elif action == 'newchat': handle_new_chat()
            elif action == 'rate': await handle_rate(data)
            elif action == 'resolved': await handle_resolved()
            elif action == 'login': await handle_login()
            elif action == 'edit-ticket':
                if state.get('pending_ticket'):
                    t = state['pending_ticket']
                    # Show edit form via JS
                    subj = t.get('subject','').replace("'","\\'")
                    desc = t.get('description','').replace("'","\\'")
                    cat = t.get('category','כללי')
                    pri = t.get('priority','בינונית')
                    await ui.run_javascript(f"""(function(){{
                        var cm=document.getElementById('chatMessages');if(!cm)return;
                        var last=cm.lastElementChild;if(last)last.innerHTML='<div style="padding:20px;background:#EFF6FF;border-radius:16px;border:1px solid #BFDBFE;direction:rtl">'
                            +'<div style="font-size:15px;font-weight:700;color:#1E40AF;margin-bottom:16px">✏️ עריכת קריאה</div>'
                            +'<div style="margin-bottom:10px"><label style="font-size:12px;color:#6B7280;font-weight:600;display:block;margin-bottom:4px">📋 נושא</label><input id="editSubj" value=\\'{subj}\\' style="width:100%;padding:10px;border-radius:8px;border:1px solid #D1D5DB;font-family:inherit;font-size:13px;direction:rtl"/></div>'
                            +'<div style="margin-bottom:10px"><label style="font-size:12px;color:#6B7280;font-weight:600;display:block;margin-bottom:4px">📝 תיאור</label><textarea id="editDesc" style="width:100%;padding:10px;border-radius:8px;border:1px solid #D1D5DB;font-family:inherit;font-size:13px;direction:rtl;min-height:60px">{desc}</textarea></div>'
                            +'<div style="display:flex;gap:10px;margin-bottom:10px">'
                            +'<div style="flex:1"><label style="font-size:12px;color:#6B7280;font-weight:600;display:block;margin-bottom:4px">📁 תחום</label><select id="editCat" style="width:100%;padding:10px;border-radius:8px;border:1px solid #D1D5DB;font-family:inherit;font-size:13px"><option>סיסמאות</option><option>Office</option><option>רשת</option><option>Moodle</option><option>הדפסה</option><option>כללי</option></select></div>'
                            +'<div style="flex:1"><label style="font-size:12px;color:#6B7280;font-weight:600;display:block;margin-bottom:4px">⚡ דחיפות</label><select id="editPri" style="width:100%;padding:10px;border-radius:8px;border:1px solid #D1D5DB;font-family:inherit;font-size:13px"><option>קריטית</option><option>דחופה</option><option>בינונית</option><option>נמוכה</option></select></div>'
                            +'</div>'
                            +'<button style="padding:12px 28px;border-radius:10px;border:none;background:#059669;color:#fff;font-size:14px;font-weight:600;cursor:pointer;font-family:inherit" data-action="submit-edit" data-value="1">✅ שלח קריאה מעודכנת</button>'
                            +'</div>';
                        var cs=document.getElementById('editCat');if(cs)cs.value='{cat}';
                        var ps=document.getElementById('editPri');if(ps)ps.value='{pri}';
                    }})()""")
            elif action == 'submit-edit':
                if state.get('pending_ticket'):
                    subj = await ui.run_javascript("document.getElementById('editSubj')?.value||''")
                    desc = await ui.run_javascript("document.getElementById('editDesc')?.value||''")
                    cat = await ui.run_javascript("document.getElementById('editCat')?.value||'כללי'")
                    pri = await ui.run_javascript("document.getElementById('editPri')?.value||'בינונית'")
                    # Submit ticket directly
                    try:
                        t_num = f"T-{str(uuid4())[:4].upper()}"
                        supabase_client.table("tickets").insert({"ticket_number": t_num, "subject": subj, "category": cat, "priority": pri, "status": "open", "student_name": state['student_name'], "phone": state.get('student_phone', ''), "ai_summary": desc, "confidence_score": 2, "message_count": len([m for m in state['chat_messages'] if m['role']=='user']), "last_update": "ממתין לטיפול טכנאי"}).execute()
                        supabase_client.table("activity_log").insert({"text": f"קריאה {t_num} נפתחה - {subj}", "icon_type": "red"}).execute()
                        state['chat_messages'][-1] = {'role':'bot','content': f"✅ קריאת שירות {t_num} נפתחה בהצלחה!\n\n📋 נושא: {subj}\n📁 קטגוריה: {cat}\n🔺 עדיפות: {pri}\n\nהטכנאי יצור קשר בהקדם.", 'suggestions': ["חזרה לדף הבית 🏠"]}
                        show_toast('✅ קריאה נפתחה בהצלחה', '#059669')
                    except Exception as e:
                        state['chat_messages'][-1] = {'role':'bot','content': f"שגיאה בפתיחת קריאה: {str(e)[:80]}"}
                    state['pending_ticket'] = None; state['pending_tc_id'] = None
                    state['needs_render'] = True
            elif action == 'filter-tickets':
                state['ticket_filter'] = data
                render_all()
            elif action == 'update-status':
                parts = data.split('|')
                if len(parts) == 2:
                    t_num, new_status = parts
                    try:
                        status_heb = {"open":"פתוח","progress":"בטיפול","done":"נפתר"}.get(new_status, new_status)
                        supabase_client.table("tickets").update({"status": new_status, "last_update": f"סטטוס עודכן ל-{status_heb}"}).eq("ticket_number", t_num).execute()
                        supabase_client.table("activity_log").insert({"text": f"קריאה {t_num} עודכנה ל-{status_heb}", "icon_type": "green"}).execute()
                        show_toast(f'✅ קריאה {t_num} עודכנה ל-{status_heb}', '#059669')
                    except Exception as e:
                        show_toast(f'❌ שגיאה בעדכון: {str(e)[:50]}', '#DC2626')
                    render_all()

    ui.timer(0.3, check_bridge)
    render_all()

try:
    ui.run(port=8090, show=False, reload=False, title='IT Support - Azrieli', storage_secret='azrieli-it-2026')
except RuntimeError as e:
    if 'Cannot add middleware' in str(e):
        print("⚠️ השרת כבר רץ! אם צריך לאתחל מחדש, עשה: Runtime > Disconnect and delete runtime")
    else:
        raise e

In [ ]:
# תא 17 - עצירת המערכת (הריצו כשסיימתם)
# tunnel.terminate()
# print("✅ System stopped")